# Multi-Object Tracking and State Estimation

**Module 01 · Perception and Prediction** · Notebook 7 of 9

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milinpatel07/Autonomous-Driving_AI-Safety-and-Security/blob/main/01_Perception/07_multi_object_tracking.ipynb)

| Level | Prerequisites | Standards |
|---|---|---|
| Intermediate | [04 Object detection](04_object_detection.ipynb), [06 Sensor fusion](06_sensor_fusion.ipynb); basic linear algebra and probability | ISO 21448:2022 (context: performance insufficiencies, triggering conditions) |

**Data:** KITTI tracking benchmark, sequence 0015 (real camera images, labels with track IDs, GPS/IMU), used in Section 9. Sections 1–8 use a labelled synthetic scene to isolate each mechanism.

**Author:** Milin Patel · Hochschule Kempten · MIT Licence

## Learning objectives

By the end of this notebook you will be able to:

1. Explain why a detector alone is not enough, and why **temporal consistency** of object hypotheses is a safety property.
2. Derive and implement a **constant-velocity Kalman filter** (predict, update, covariance propagation) from scratch in NumPy.
3. Check filter **consistency** with the NEES and NIS statistics and recognise a mistuned filter.
4. Implement **data association** with Mahalanobis gating and optimal (Hungarian) assignment.
5. Implement **track management** (tentative, confirmed, deleted; M-of-N confirmation; coasting through occlusion).
6. Evaluate a tracker with **MOTA, ID switches and HOTA**, and explain what AMOTA adds.
7. Quantify how **detection dropout and noise** delay track confirmation and break track continuity, and relate this to SOTIF triggering conditions.
8. Run the tracker on a **real KITTI sequence** with a pretrained detector, score it against the real track IDs, and trace real identity switches to occlusion.

## 1. Why tracking matters for safety

An object detector answers *"what is where, in this frame?"*. Motion planning needs more: *"where has this object been, how fast is it moving, and where will it be in two seconds?"* Answering that requires associating detections across time into **tracks** and estimating a dynamic state (position, velocity, possibly acceleration and heading) with its uncertainty. A tracker therefore provides three things a single-frame detector cannot:

- **Velocity and intent.** Velocity is not observable from one position measurement; it emerges only from a sequence.
- **Robustness to missed detections.** A track can be *predicted* ("coasted") through a short occlusion or a missed frame.
- **Suppression of false positives.** Requiring several consistent detections before reporting an object filters out isolated clutter.

Each of these is also a source of risk: discarding history loses velocity; coasting too long keeps ghosts alive; confirming too slowly delays the reaction.

### A case in point: Tempe, Arizona, 18 March 2018

The U.S. National Transportation Safety Board (NTSB) investigated the fatal collision between an Uber ATG automated test vehicle and a pedestrian (accident ID **HWY18MH010**; report **NTSB/HAR-19/03**, adopted 19 November 2019). The report states that the automated driving system (ADS) first detected the pedestrian **5.6 seconds** before impact, but changed her classification several times, alternating between vehicle, bicycle and *other*. On tracking history, the report says (Section 1.5.5.2, p. 12):

> *"If the perception process changed the classification of a detected object, it no longer considered the tracking history of that object when generating a new trajectory."*

and concludes (p. 16–17) that, because the system never classified her as a pedestrian *"and the system's design excluded tracking history for nonpersisting objects—those with changed classifications—it was unable to correctly predict the pedestrian's path."* The ADS recognised an emergency only **1.2 seconds** before impact. The report also records that Uber ATG subsequently changed the system so that it *"retains tracking history even if detected objects are reclassified"* (p. 13).

The probable cause determined by the NTSB centred on the vehicle operator's failure to monitor the driving environment, with contributing factors including Uber ATG's inadequate safety risk assessment and oversight. The tracking design is therefore one factor among several, but it is a clear, documented example of a **perception-related functional insufficiency** in the sense of ISO 21448:2022: no component failed, yet the intended functionality was insufficient for a foreseeable situation (a pedestrian pushing a bicycle across a road away from a crosswalk). In Section 7 we reproduce a *simplified analogue* of this design choice and measure its effect.

> **Note:** our simulation is a teaching analogue; it is not a reconstruction of the Uber ATG software.

In [ ]:
# Course toolkit (tables, formulas and real-data loaders). Installs automatically on Colab.
try:
    import avsafety
except ImportError:
    %pip install -q "avsafety[data] @ git+https://github.com/milinpatel07/Autonomous-Driving_AI-Safety-and-Security.git"
    import avsafety
from avsafety import data as D

import itertools
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse, Rectangle
from scipy.optimize import linear_sum_assignment
from scipy.stats import chi2

SEED = 42
rng = np.random.default_rng(SEED)

# Okabe–Ito colour-blind-safe palette
OI = {"orange": "#E69F00", "skyblue": "#56B4E9", "green": "#009E73", "yellow": "#F0E442",
      "blue": "#0072B2", "vermillion": "#D55E00", "purple": "#CC79A7", "black": "#000000"}
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.titleweight": "bold"})
T0 = time.time()
print("NumPy", np.__version__, "| setup complete")

## 2. The constant-velocity Kalman filter, step by step

### 2.1 State-space model

We track a point object in a 2D ground-plane frame with state

$$\mathbf{x}_k = \begin{bmatrix} p_x & p_y & v_x & v_y \end{bmatrix}^\top .$$

The **constant-velocity (CV) model** treats acceleration as zero-mean white noise $a_k \sim \mathcal{N}(0, \sigma_a^2)$ per axis, held constant over one sampling interval $\Delta t$ (the *discrete white-noise acceleration* model; Bar-Shalom et al., 2001):

$$\mathbf{x}_{k} = \mathbf{F}\,\mathbf{x}_{k-1} + \mathbf{w}_{k},\qquad
\mathbf{F} = \begin{bmatrix} 1 & 0 & \Delta t & 0\\ 0 & 1 & 0 & \Delta t\\ 0&0&1&0\\0&0&0&1 \end{bmatrix},\qquad
\mathbf{w}_k \sim \mathcal{N}(\mathbf{0}, \mathbf{Q}).$$

Because the noise enters through $\boldsymbol{\Gamma} = [\tfrac{1}{2}\Delta t^2,\ \Delta t]^\top$ on each axis, $\mathbf{Q} = \sigma_a^2\,\boldsymbol{\Gamma}\boldsymbol{\Gamma}^\top$ per axis:

$$\mathbf{Q}_{\text{axis}} = \sigma_a^2 \begin{bmatrix} \tfrac{1}{4}\Delta t^4 & \tfrac{1}{2}\Delta t^3 \\ \tfrac{1}{2}\Delta t^3 & \Delta t^2 \end{bmatrix}.$$

A detector (camera, LiDAR or radar after clustering) measures position only:

$$\mathbf{z}_k = \mathbf{H}\mathbf{x}_k + \mathbf{v}_k,\qquad \mathbf{H} = \begin{bmatrix} 1&0&0&0\\0&1&0&0\end{bmatrix},\qquad \mathbf{v}_k\sim\mathcal{N}(\mathbf{0}, \mathbf{R}=\sigma_z^2\mathbf{I}_2).$$

### 2.2 Predict

Propagate the mean and covariance through the linear model:

$$\hat{\mathbf{x}}_{k|k-1} = \mathbf{F}\hat{\mathbf{x}}_{k-1|k-1}, \qquad
\mathbf{P}_{k|k-1} = \mathbf{F}\mathbf{P}_{k-1|k-1}\mathbf{F}^\top + \mathbf{Q}.$$

Uncertainty *grows* during prediction. During an occlusion we can only predict, so the covariance inflates every step: this is what makes coasting possible but also what eventually makes it meaningless.

### 2.3 Update

The **innovation** (measurement residual) and its covariance are

$$\boldsymbol{\nu}_k = \mathbf{z}_k - \mathbf{H}\hat{\mathbf{x}}_{k|k-1},\qquad \mathbf{S}_k = \mathbf{H}\mathbf{P}_{k|k-1}\mathbf{H}^\top + \mathbf{R}.$$

The **Kalman gain** weighs prediction against measurement,

$$\mathbf{K}_k = \mathbf{P}_{k|k-1}\mathbf{H}^\top\mathbf{S}_k^{-1},$$

and the posterior is

$$\hat{\mathbf{x}}_{k|k} = \hat{\mathbf{x}}_{k|k-1} + \mathbf{K}_k\boldsymbol{\nu}_k,\qquad
\mathbf{P}_{k|k} = (\mathbf{I}-\mathbf{K}_k\mathbf{H})\mathbf{P}_{k|k-1}(\mathbf{I}-\mathbf{K}_k\mathbf{H})^\top + \mathbf{K}_k\mathbf{R}\mathbf{K}_k^\top .$$

The last expression is the **Joseph form**; it is algebraically equal to $(\mathbf{I}-\mathbf{K}\mathbf{H})\mathbf{P}$ but stays symmetric and positive semi-definite under rounding, which matters in long-running embedded filters. For linear-Gaussian models this recursion is the minimum mean-square-error estimator (Kalman, 1960).

### 2.4 Consistency: NEES and NIS

A filter is **consistent** if its reported covariance matches its actual errors. Two standard tests (Bar-Shalom et al., 2001):

- **NEES** (normalised estimation error squared), requires ground truth (simulation):
  $\epsilon_k = (\mathbf{x}_k-\hat{\mathbf{x}}_{k|k})^\top \mathbf{P}_{k|k}^{-1}(\mathbf{x}_k-\hat{\mathbf{x}}_{k|k}) \sim \chi^2_{n}$, with $n=4$.
- **NIS** (normalised innovation squared), computable *online* without ground truth:
  $\epsilon_{\nu,k} = \boldsymbol{\nu}_k^\top \mathbf{S}_k^{-1}\boldsymbol{\nu}_k \sim \chi^2_{m}$, with $m=2$.

Averaging over $N$ Monte Carlo runs, $N\bar{\epsilon}_k \sim \chi^2_{Nn}$, which gives a two-sided 95 % acceptance interval. NEES persistently **above** the interval means the filter is **over-confident** (its covariance is too small): the dangerous direction, because gates become too tight and downstream planners trust a position that is wrong. NIS is the quantity a deployed system can monitor at runtime.

In [ ]:
def cv_model(dt, sigma_a):
    # Return F, Q for the 2D discrete white-noise-acceleration CV model, state [px, py, vx, vy]
    F = np.array([[1, 0, dt, 0], [0, 1, 0, dt], [0, 0, 1, 0], [0, 0, 0, 1]], dtype=float)
    q = sigma_a**2 * np.array([[dt**4 / 4, dt**3 / 2], [dt**3 / 2, dt**2]])
    Q = np.zeros((4, 4))
    Q[np.ix_([0, 2], [0, 2])] = q      # x-axis block (px, vx)
    Q[np.ix_([1, 3], [1, 3])] = q      # y-axis block (py, vy)
    return F, Q

H = np.array([[1, 0, 0, 0], [0, 1, 0, 0]], dtype=float)


class KalmanCV:
    # Linear Kalman filter with a constant-velocity motion model and position measurements
    def __init__(self, x0, P0, dt=0.1, sigma_a=1.0, sigma_z=0.3):
        self.x = np.asarray(x0, dtype=float).copy()
        self.P = np.asarray(P0, dtype=float).copy()
        self.F, self.Q = cv_model(dt, sigma_a)
        self.R = sigma_z**2 * np.eye(2)

    def predict(self):
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q
        return self.x

    def innovation(self, z, R=None):
        # R: optional per-measurement noise covariance (e.g. range-dependent); default self.R
        nu = np.asarray(z, dtype=float) - H @ self.x
        S = H @ self.P @ H.T + (self.R if R is None else R)
        return nu, S

    def mahalanobis2(self, z):
        nu, S = self.innovation(z)
        return float(nu @ np.linalg.solve(S, nu))

    def update(self, z, R=None):
        R = self.R if R is None else R
        nu, S = self.innovation(z, R)
        K = self.P @ H.T @ np.linalg.inv(S)
        self.x = self.x + K @ nu
        IKH = np.eye(4) - K @ H
        self.P = IKH @ self.P @ IKH.T + K @ R @ K.T          # Joseph form
        return float(nu @ np.linalg.solve(S, nu))             # NIS of this update


F_demo, Q_demo = cv_model(0.1, 1.0)
print("F =\n", F_demo)
print("Q (sigma_a = 1 m/s^2, dt = 0.1 s) =\n", np.round(Q_demo, 6))

### 2.5 A single target, three tunings

We simulate a target whose true motion follows the CV model with $\sigma_{a,\text{true}} = 0.5\ \text{m/s}^2$ (a gently manoeuvring cyclist), measured at 10 Hz with $\sigma_z = 0.5$ m. We run three filters that differ only in their assumed process noise $\sigma_a$:

- **matched** ($\sigma_a = 0.5$), **over-confident** ($\sigma_a = 0.05$, believes the target barely manoeuvres), and **conservative** ($\sigma_a = 5$).

> **Illustrative data:** values below are synthetic, chosen for teaching; they are not measurements.

In [ ]:
def simulate_cv_truth(n_steps, dt, sigma_a, rng, x0=(0.0, 0.0, 4.0, 1.0)):
    F, Q = cv_model(dt, sigma_a)
    L = np.linalg.cholesky(Q + 1e-12 * np.eye(4))
    X = np.zeros((n_steps, 4))
    X[0] = x0
    for k in range(1, n_steps):
        X[k] = F @ X[k - 1] + L @ rng.standard_normal(4)
    return X


def run_filter(X, Z, dt, sigma_a, sigma_z):
    P0 = np.diag([sigma_z**2, sigma_z**2, 4.0, 4.0])
    x0 = np.array([Z[0, 0], Z[0, 1], 0.0, 0.0])
    kf = KalmanCV(x0, P0, dt, sigma_a, sigma_z)
    est, covs, nees, nis = [kf.x.copy()], [kf.P.copy()], [np.nan], [np.nan]
    for k in range(1, len(Z)):
        kf.predict()
        nis.append(kf.update(Z[k]))
        e = X[k] - kf.x
        nees.append(float(e @ np.linalg.solve(kf.P, e)))
        est.append(kf.x.copy()); covs.append(kf.P.copy())
    return np.array(est), np.array(covs), np.array(nees), np.array(nis)


DT, SIG_A_TRUE, SIG_Z, N_STEPS, N_MC = 0.1, 0.5, 0.5, 120, 50
TUNINGS = {"over-confident (sigma_a=0.05)": 0.05, "matched (sigma_a=0.5)": 0.5,
           "conservative (sigma_a=5)": 5.0}
TUNE_COL = dict(zip(TUNINGS, [OI["vermillion"], OI["blue"], OI["orange"]]))

mc_rng = np.random.default_rng(SEED)
nees_mc = {name: np.zeros((N_MC, N_STEPS)) for name in TUNINGS}
nis_mc = {name: np.zeros((N_MC, N_STEPS)) for name in TUNINGS}
rmse_pos = {name: [] for name in TUNINGS}
for r in range(N_MC):
    X = simulate_cv_truth(N_STEPS, DT, SIG_A_TRUE, mc_rng)
    Z = X[:, :2] + SIG_Z * mc_rng.standard_normal((N_STEPS, 2))
    if r == 0:
        X_demo, Z_demo = X, Z
    for name, sa in TUNINGS.items():
        est, _, nees, nis = run_filter(X, Z, DT, sa, SIG_Z)
        nees_mc[name][r], nis_mc[name][r] = nees, nis
        rmse_pos[name].append(np.sqrt(np.mean(np.sum((est[20:, :2] - X[20:, :2])**2, axis=1))))

# 95 % two-sided bounds for the MC-averaged statistics
nees_lo, nees_hi = chi2.ppf([0.025, 0.975], N_MC * 4) / N_MC
nis_lo, nis_hi = chi2.ppf([0.025, 0.975], N_MC * 2) / N_MC
print(f"NEES 95% interval for the {N_MC}-run average: [{nees_lo:.2f}, {nees_hi:.2f}] (expected value 4)")
print(f"NIS  95% interval for the {N_MC}-run average: [{nis_lo:.2f}, {nis_hi:.2f}] (expected value 2)")

In [ ]:
def cov_ellipse(ax, mean, cov, n_sigma=2.0, **kw):
    vals, vecs = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(vecs[1, 1], vecs[0, 1]))
    w, h = 2 * n_sigma * np.sqrt(np.maximum(vals[::-1], 0))
    ax.add_patch(Ellipse(mean, w, h, angle=angle, fill=False, **kw))


fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
ax = axes[0]
ax.plot(X_demo[:, 0], X_demo[:, 1], color=OI["black"], lw=2, label="ground truth")
ax.scatter(Z_demo[:, 0], Z_demo[:, 1], s=8, color="grey", alpha=0.6, label="measurements")
for name in ["over-confident (sigma_a=0.05)", "matched (sigma_a=0.5)"]:
    est, covs, _, _ = run_filter(X_demo, Z_demo, DT, TUNINGS[name], SIG_Z)
    ax.plot(est[:, 0], est[:, 1], color=TUNE_COL[name], lw=1.5, label=name)
    for k in range(10, N_STEPS, 25):
        cov_ellipse(ax, est[k, :2], covs[k][:2, :2], color=TUNE_COL[name], lw=1)
ax.set(title="Single run: estimates with 2-sigma ellipses", xlabel="x [m]", ylabel="y [m]")
ax.set_aspect("equal", adjustable="datalim"); ax.legend(fontsize=8)

t_axis = np.arange(N_STEPS) * DT
for ax, stat, lo, hi, dof, lab in [(axes[1], nees_mc, nees_lo, nees_hi, 4, "NEES"),
                                   (axes[2], nis_mc, nis_lo, nis_hi, 2, "NIS")]:
    for name in TUNINGS:
        ax.plot(t_axis[1:], stat[name][:, 1:].mean(axis=0), color=TUNE_COL[name], label=name)
    ax.axhspan(lo, hi, color=OI["green"], alpha=0.2, label="95 % acceptance band")
    ax.axhline(dof, color=OI["green"], ls="--", lw=1)
    ax.set_yscale("log")
    ax.set(title=f"{lab}, average over {N_MC} Monte Carlo runs", xlabel="time [s]",
           ylabel=f"average {lab} [-] (log scale)")
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

rows = []
for name in TUNINGS:
    a_nees = nees_mc[name][:, 1:].mean(axis=0)
    a_nis = nis_mc[name][:, 1:].mean(axis=0)
    rows.append({"tuning": name,
                 "mean NEES": a_nees[20:].mean(),
                 "NEES inside band [%]": 100 * np.mean((a_nees[20:] >= nees_lo) & (a_nees[20:] <= nees_hi)),
                 "mean NIS": a_nis[20:].mean(),
                 "NIS inside band [%]": 100 * np.mean((a_nis[20:] >= nis_lo) & (a_nis[20:] <= nis_hi)),
                 "position RMSE [m]": np.mean(rmse_pos[name])})
consistency = pd.DataFrame(rows).set_index("tuning")
print(consistency.round(2).to_string())
worst = consistency["mean NEES"].idxmax()
best = (consistency["mean NEES"] - 4).abs().idxmin()
print(f"\nText summary: '{best}' is closest to the expected NEES of 4 "
      f"({consistency.loc[best, 'mean NEES']:.2f}); '{worst}' has mean NEES "
      f"{consistency.loc[worst, 'mean NEES']:.1f}, i.e. its covariance under-states the real error "
      f"by a factor of about {consistency.loc[worst, 'mean NEES'] / 4:.1f} in squared terms.")

**Reading the plots.** The over-confident filter produces a smooth trajectory with tiny ellipses that frequently do not contain the truth, and its NEES lies far above the band. The conservative filter is *consistent in the safe direction* (NEES below the band: it over-states its uncertainty) but tracks noisily. Consistency and accuracy are different properties; a safety argument usually needs both, and over-confidence is the failure mode to rule out first, because gating, prediction and planning all consume the covariance.

## 3. Data association: which detection belongs to which track?

With several objects and clutter, each frame poses an assignment problem between $T$ predicted tracks and $D$ detections.

### 3.1 Gating with the Mahalanobis distance

For track $i$ with predicted measurement $\mathbf{H}\hat{\mathbf{x}}_i$ and innovation covariance $\mathbf{S}_i$, the squared Mahalanobis distance of detection $j$ is

$$d^2_{ij} = (\mathbf{z}_j - \mathbf{H}\hat{\mathbf{x}}_i)^\top\mathbf{S}_i^{-1}(\mathbf{z}_j - \mathbf{H}\hat{\mathbf{x}}_i).$$

If detection $j$ really originates from track $i$, $d^2_{ij}\sim\chi^2_2$. A **gate** $d^2_{ij} \le \gamma$ with $\gamma = \chi^2_{2}(0.99) \approx 9.21$ keeps the true detection with probability 0.99 and discards implausible pairs. The gate scales automatically with uncertainty: a coasting track has a large $\mathbf{S}_i$ and therefore a large gate.

### 3.2 Global nearest neighbour via the Hungarian algorithm

Among gated pairs we minimise the total cost $\sum_{(i,j)} c_{ij}$ subject to one-to-one assignment. A common cost is the negative log-likelihood up to a constant, $c_{ij} = d^2_{ij} + \ln\lvert\mathbf{S}_i\rvert$. The optimal solution is found in polynomial time by the Hungarian method (Kuhn, 1955; Munkres, 1957); SciPy provides it as `scipy.optimize.linear_sum_assignment`. Pairs outside the gate receive a prohibitive cost and are rejected after the assignment.

The example below shows why a **greedy** nearest-neighbour rule (take the globally closest pair first, then the next closest among the remaining, ...) can leave a track without an update even though a complete, gate-respecting assignment exists.

In [ ]:
GATE = chi2.ppf(0.99, df=2)
BIG = 1e6


def associate(tracks_pred, S_list, dets, gate=GATE, R_dets=None):
    # Global nearest neighbour: returns matches [(i, j)], unmatched track indices, unmatched detection indices.
    # With R_dets (one covariance per detection), S_list holds H P H^T and S_ij = H P_i H^T + R_j.
    T, D = len(tracks_pred), len(dets)
    if T == 0 or D == 0:
        return [], list(range(T)), list(range(D))
    cost = np.full((T, D), BIG)
    for i in range(T):
        for j in range(D):
            S = S_list[i] if R_dets is None else S_list[i] + R_dets[j]
            nu = dets[j] - tracks_pred[i]
            d2 = nu @ np.linalg.solve(S, nu)
            if d2 <= gate:
                cost[i, j] = d2 + np.log(np.linalg.det(S))
    rows, cols = linear_sum_assignment(cost)
    matches = [(int(i), int(j)) for i, j in zip(rows, cols) if cost[i, j] < BIG]
    um_t = sorted(set(range(T)) - {i for i, _ in matches})
    um_d = sorted(set(range(D)) - {j for _, j in matches})
    return matches, um_t, um_d


def associate_greedy(tracks_pred, S_list, dets, gate=GATE):
    pairs = []
    for i, j in itertools.product(range(len(tracks_pred)), range(len(dets))):
        nu = dets[j] - tracks_pred[i]
        d2 = nu @ np.linalg.solve(S_list[i], nu)
        if d2 <= gate:
            pairs.append((d2, i, j))
    used_t, used_d, matches = set(), set(), []
    for d2, i, j in sorted(pairs):
        if i not in used_t and j not in used_d:
            matches.append((i, j)); used_t.add(i); used_d.add(j)
    return matches


# Two tracks close together, two detections, plus one clutter detection far away
pred = [np.array([0.0, 0.0]), np.array([2.0, 0.0])]
S_ex = [0.6 * np.eye(2), 0.6 * np.eye(2)]
dets_ex = [np.array([1.05, 0.15]), np.array([3.4, 0.2]), np.array([1.0, 4.0])]

hung, _, _ = associate(pred, S_ex, dets_ex)
greedy = associate_greedy(pred, S_ex, dets_ex)

def total_d2(matches):
    return sum((dets_ex[j] - pred[i]) @ np.linalg.solve(S_ex[i], dets_ex[j] - pred[i]) for i, j in matches)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, matches, title in [(axes[0], greedy, "Greedy nearest neighbour"), (axes[1], hung, "Hungarian (global optimum)")]:
    for i, p in enumerate(pred):
        ax.scatter(*p, marker="s", s=90, color=OI["blue"], zorder=3)
        ax.annotate(f"track {i}", p + [0.0, -0.55], ha="center")
        cov_ellipse(ax, p, S_ex[i], n_sigma=np.sqrt(GATE), color=OI["blue"], ls="--", lw=1)
    for j, d in enumerate(dets_ex):
        ax.scatter(*d, marker="x", s=90, color=OI["vermillion"], zorder=3)
        ax.annotate(f"det {j}", d + [0.1, 0.25])
    for i, j in matches:
        ax.plot(*zip(pred[i], dets_ex[j]), color=OI["green"], lw=2.5)
    ax.set(title=f"{title}: tracks updated = {len(matches)}", xlabel="x [m]", ylabel="y [m]", xlim=(-3, 5.5), ylim=(-3, 5))
    ax.set_aspect("equal")
plt.suptitle("Gates (dashed: 99 % chi-square gate) and assignments", fontweight="bold")
plt.tight_layout(); plt.show()
for name, mt in [("Greedy   ", greedy), ("Hungarian", hung)]:
    print(f"{name}: matches (track, det) = {mt}, tracks updated = {len(mt)} of {len(pred)}, "
          f"sum of d^2 over matches = {total_d2(mt):.2f}")
print("Text summary: greedy updates", len(greedy), "track(s); the Hungarian solution updates", len(hung),
      "because an unmatched gated track is penalised with the prohibitive cost BIG.")

The greedy rule gives detection 0 to whichever track is marginally closer (here track 1) and thereby leaves track 0 without a partner, although the pairing (0→0, 1→1) respects both gates. Its smaller sum of $d^2$ is not a merit: it simply contains fewer matches. Missed updates of this kind make a track coast unnecessarily and, in dense urban scenes, lead to **identity swaps** between nearby pedestrians. Detection 2 lies outside both gates and is correctly treated as a candidate for a new track (or as clutter).

More elaborate schemes avoid committing to a single hard assignment: the joint probabilistic data association filter (JPDA) weights all gated detections, and multiple hypothesis tracking (MHT) defers the decision over several frames. Both are covered in Bar-Shalom et al. (2001) and remain in use in radar-centric automotive trackers.

## 4. Track management

Association decides *which* detection updates *which* track. Track management decides **when a track is born, when it is believed, and when it dies**:

| State | Entered when | Reported to planning? |
|---|---|---|
| **Tentative** | an unassigned detection spawns a new track | no |
| **Confirmed** | the track was associated in at least $M$ of its last $N$ frames (**M-of-N logic**) | yes |
| **Coasting** (confirmed, missed) | a confirmed track receives no detection; it is predicted only | yes, with growing covariance |
| **Deleted** | a tentative track fails to reach $M$ of $N$, or a confirmed track misses more than $K_{\max}$ consecutive frames, or leaves the field of view | no |

The parameters $M$, $N$ and $K_{\max}$ encode a **safety trade-off**:

- larger $M$ suppresses clutter (fewer false tracks, fewer phantom-braking events) but **delays confirmation** of real objects;
- larger $K_{\max}$ bridges longer occlusions (fewer identity switches, preserved velocity) but keeps **ghost tracks** alive longer after an object has gone.

The tracker below combines Sections 2–4. Its process noise defaults to $\sigma_a = 3$ m/s²: a single CV model must be given enough process noise to cover the hardest manoeuvre it should follow (here a car braking at about 2.6 m/s²), at the price of noisier estimates for gently moving objects; Section 11 discusses the alternative. It is deliberately compact (about 90 lines) so it can be read in full. Two options are used only in the real-data part (Section 9): per-detection noise covariances `det_R` (range-dependent sensor noise) and `fov=None` (tracking in a world frame, where the field-of-view test is replaced by the coasting limit). It also carries a class label per track, and the option `reset_on_reclass` reproduces the design choice discussed in Section 1: if the associated detection's class differs from the track's class, the track is discarded and a new one is started without history.

In [ ]:
class Track:
    _next_id = 0

    def __init__(self, z, cls, cfg, R=None):
        self.id = Track._next_id; Track._next_id += 1
        P0 = np.diag([cfg["sigma_z"]**2, cfg["sigma_z"]**2, cfg["v0_var"], cfg["v0_var"]])
        if R is not None:
            P0[:2, :2] = R                       # initial position uncertainty = this detection's noise
        self.kf = KalmanCV([z[0], z[1], 0.0, 0.0], P0, cfg["dt"], cfg["sigma_a"], cfg["sigma_z"])
        self.state = "tentative"
        self.hits = [True]               # association history, most recent last
        self.misses = 0                  # consecutive misses
        self.class_votes = {cls: 1}

    @property
    def cls(self):
        return max(self.class_votes, key=self.class_votes.get)


class MultiObjectTracker:
    # GNN tracker: CV Kalman filter + chi-square gating + Hungarian assignment + M-of-N management
    def __init__(self, dt=0.1, sigma_a=3.0, sigma_z=0.3, M=3, N=5, max_coast=8,
                 v0_var=100.0, reset_on_reclass=False, fov=((0.0, 60.0), (-20.0, 20.0))):
        self.cfg = dict(dt=dt, sigma_a=sigma_a, sigma_z=sigma_z, v0_var=v0_var)
        self.M, self.N, self.max_coast = M, N, max_coast
        self.reset_on_reclass, self.fov = reset_on_reclass, fov
        self.tracks = []
        self.n_reclass_resets = 0

    def _in_fov(self, p):
        if self.fov is None:
            return True
        (x0, x1), (y0, y1) = self.fov
        return x0 <= p[0] <= x1 and y0 <= p[1] <= y1

    def step(self, det_pos, det_cls, det_R=None):
        Rj = (lambda j: None) if det_R is None else (lambda j: det_R[j])
        # 1) predict all tracks
        for t in self.tracks:
            t.kf.predict()
        # 2) associate
        preds = [H @ t.kf.x for t in self.tracks]
        if det_R is None:
            S_list = [t.kf.innovation(np.zeros(2))[1] for t in self.tracks]
        else:
            S_list = [H @ t.kf.P @ H.T for t in self.tracks]
        matches, um_t, um_d = associate(preds, S_list, det_pos, R_dets=det_R)
        # 3) update matched tracks
        new_tracks = []
        for i, j in matches:
            t = self.tracks[i]
            if self.reset_on_reclass and det_cls[j] != t.cls:
                t.state = "deleted"                       # history discarded on reclassification
                new_tracks.append(Track(det_pos[j], det_cls[j], self.cfg, Rj(j)))
                self.n_reclass_resets += 1
                continue
            t.kf.update(det_pos[j], Rj(j))
            t.hits.append(True); t.misses = 0
            t.class_votes[det_cls[j]] = t.class_votes.get(det_cls[j], 0) + 1
        for i in um_t:
            t = self.tracks[i]
            t.hits.append(False); t.misses += 1
        # 4) track management
        for t in self.tracks:
            if t.state == "deleted":
                continue
            recent = t.hits[-self.N:]
            if t.state == "tentative":
                if sum(recent) >= self.M:
                    t.state = "confirmed"
                elif len(t.hits) >= self.N or t.misses > self.N - self.M:
                    t.state = "deleted"
            elif t.misses > self.max_coast:
                t.state = "deleted"
            if not self._in_fov(t.kf.x[:2]):
                t.state = "deleted"
        # 5) births from unassigned detections
        new_tracks += [Track(det_pos[j], det_cls[j], self.cfg, Rj(j)) for j in um_d]
        self.tracks = [t for t in self.tracks if t.state != "deleted"] + new_tracks
        # output: confirmed tracks only
        return {t.id: (t.kf.x.copy(), t.kf.P.copy(), t.cls, t.misses)
                for t in self.tracks if t.state == "confirmed"}


print("Tracker defined. Default gate gamma = chi2_2(0.99) =", round(GATE, 2))

### 4.1 How quickly does M-of-N confirm an object?

Before running the full scene, we can compute the confirmation latency analytically. Suppose an object is detected independently in each frame with probability $P_D$. A track is born on the first detection; it is confirmed at the first frame at which $M$ of the last $N$ frames contain a detection. We enumerate all detection sequences to obtain the distribution of the time-to-confirm and the probability that the tentative track dies before confirmation (after which the process restarts). This dependence of latency on $P_D$ is the bridge to SOTIF: **any triggering condition that lowers $P_D$** (rain, low light, partial occlusion, unusual appearance) **lengthens the time before planning is told the object exists**.

In [ ]:
def confirm_latency_mc(p_d, M, N, n_runs=800, max_frames=200, seed=SEED):
    # Frames from first detection opportunity to confirmation, with tentative-track deletion and restart
    r = np.random.default_rng(seed)
    out = np.empty(n_runs)
    for k in range(n_runs):
        hist, frame = None, 0
        while frame < max_frames:
            hit = r.random() < p_d
            if hist is None:
                if hit:
                    hist = [True]
            else:
                hist.append(hit)
                misses = 0
                for h in reversed(hist):
                    if h:
                        break
                    misses += 1
                if sum(hist[-N:]) >= M:
                    break
                if len(hist) >= N or misses > N - M:
                    hist = None                     # tentative track deleted
            frame += 1
        out[k] = frame
    return out


p_grid = np.round(np.arange(0.3, 1.0001, 0.05), 2)
rules = [(2, 3), (3, 5), (4, 5)]
lat = {rule: np.array([confirm_latency_mc(p, *rule) for p in p_grid]) for rule in rules}

DT_SCENE = 0.1
v_ego = 50 / 3.6
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
cols = [OI["blue"], OI["orange"], OI["vermillion"]]
for (rule, L), c in zip(lat.items(), cols):
    mean_s = L.mean(axis=1) * DT_SCENE
    p95_s = np.percentile(L, 95, axis=1) * DT_SCENE
    axes[0].plot(p_grid, mean_s, "o-", color=c, label=f"M={rule[0]} of N={rule[1]}: mean")
    axes[0].plot(p_grid, p95_s, "--", color=c, label=f"M={rule[0]} of N={rule[1]}: 95th percentile")
    axes[1].plot(p_grid, p95_s * v_ego, "o-", color=c, label=f"M={rule[0]} of N={rule[1]}")
axes[0].set(title="Time-to-confirm vs detection probability (10 Hz)", xlabel="per-frame detection probability P_D [-]",
            ylabel="time-to-confirm [s]")
axes[0].set_yscale("log"); axes[0].legend(fontsize=8)
axes[1].set(title="Distance driven at 50 km/h before confirmation (95th pct.)",
            xlabel="per-frame detection probability P_D [-]", ylabel="distance [m]")
axes[1].set_yscale("log"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

for rule, L in lat.items():
    i9, i6 = list(p_grid).index(0.9), list(p_grid).index(0.6)
    print(f"M={rule[0]}/N={rule[1]}: 95th-pct time-to-confirm {np.percentile(L[i9], 95) * DT_SCENE:.2f} s at P_D=0.9 "
          f"-> {np.percentile(L[i6], 95) * DT_SCENE:.2f} s at P_D=0.6 "
          f"(+{(np.percentile(L[i6], 95) - np.percentile(L[i9], 95)) * DT_SCENE * v_ego:.1f} m driven at 50 km/h)")

## 5. A synthetic urban scene

We now build a small but complete scenario in the sensor frame of a stationary ego vehicle at the origin, looking along $+x$, sampled at 10 Hz for 15 s:

- a **pedestrian** crossing the road at $x = 25$ m from right to left at 1.4 m/s;
- a **parked vehicle** on the right-hand side which **occludes** the pedestrian for part of the crossing;
- a **cyclist** riding away from the ego vehicle at 5 m/s;
- an **oncoming car** decelerating from 12 m/s to stop before the crossing (a manoeuvre that the CV model does not describe exactly);
- **clutter**: Poisson-distributed false positives, uniform over the field of view;
- per-class detection probability $P_D$ and Gaussian position noise $\sigma_z$; and **label noise**: with some probability a detection carries the wrong class.

> **Illustrative data:** values below are synthetic, chosen for teaching; they are not measurements.

In [ ]:
SCENE = dict(dt=0.1, T=15.0, fov=((0.0, 60.0), (-20.0, 20.0)),
             van=((19.0, 22.0), (-6.0, -4.2)))           # parked vehicle: occluder rectangle (x-range, y-range) [m]
CLASSES = ["pedestrian", "cyclist", "vehicle", "other"]


def ground_truth(t):
    # Return dict object_id -> (position, velocity, class) at time t [s]
    gt = {}
    if t >= 1.0:                                           # pedestrian crossing at x = 25 m
        gt[0] = (np.array([25.0, -12.0 + 1.4 * (t - 1.0)]), np.array([0.0, 1.4]), "pedestrian")
    if True:                                               # cyclist riding away at 5 m/s (leaves the FOV)
        gt[1] = (np.array([4.0 + 5.0 * t, 2.5]), np.array([5.0, 0.0]), "cyclist")
    a_car, v0, x0 = -12.0**2 / (2 * 28.0), 12.0, 58.0       # stops after 28 m, at x = 30 m
    t_stop = -v0 / a_car
    tc = min(t, t_stop)
    gt[2] = (np.array([x0 - (v0 * tc + 0.5 * a_car * tc**2), -1.75]),
             np.array([-(v0 + a_car * tc), 0.0]), "vehicle")
    return gt


def occluded(p, van=SCENE["van"], n=60):
    # Line-of-sight test from the sensor at the origin to point p against the parked-vehicle rectangle
    s = np.linspace(0.0, 1.0, n)[:, None] * p[None, :]
    (x0, x1), (y0, y1) = van
    inside = (s[:, 0] >= x0) & (s[:, 0] <= x1) & (s[:, 1] >= y0) & (s[:, 1] <= y1)
    return bool(inside[:-1].any()) and not (x0 <= p[0] <= x1 and y0 <= p[1] <= y1)


def in_fov(p, fov=SCENE["fov"]):
    (x0, x1), (y0, y1) = fov
    return x0 <= p[0] <= x1 and y0 <= p[1] <= y1


def simulate_scene(p_d=None, sigma_z=0.3, clutter_rate=1.0, p_mislabel=None, seed=SEED):
    # Returns per-frame ground truth and detections. p_d / p_mislabel: dict per class or scalar
    r = np.random.default_rng(seed)
    p_d = p_d if isinstance(p_d, dict) else {c: (0.9 if p_d is None else p_d) for c in CLASSES}
    p_ml = p_mislabel if isinstance(p_mislabel, dict) else {c: (0.05 if p_mislabel is None else p_mislabel) for c in CLASSES}
    times = np.arange(0.0, SCENE["T"] + 1e-9, SCENE["dt"])
    frames = []
    for t in times:
        gt_all = ground_truth(t)
        gt = {i: g for i, g in gt_all.items() if in_fov(g[0])}
        vis = {i: not occluded(g[0]) for i, g in gt.items()}
        pos, cls, src = [], [], []
        for i, (p, v, c) in gt.items():
            if vis[i] and r.random() < p_d[c]:
                pos.append(p + sigma_z * r.standard_normal(2))
                if r.random() < p_ml[c]:
                    cls.append(r.choice([k for k in CLASSES if k != c]))
                else:
                    cls.append(c)
                src.append(i)
        (x0, x1), (y0, y1) = SCENE["fov"]
        for _ in range(r.poisson(clutter_rate)):
            pos.append(np.array([r.uniform(x0, x1), r.uniform(y0, y1)]))
            cls.append(r.choice(CLASSES)); src.append(-1)
        frames.append(dict(t=t, gt=gt, visible=vis, det_pos=pos, det_cls=cls, det_src=src))
    return frames


def run_tracker(frames, **tracker_kw):
    Track._next_id = 0
    trk = MultiObjectTracker(dt=SCENE["dt"], **tracker_kw)
    outputs = [trk.step(f["det_pos"], f["det_cls"]) for f in frames]
    return outputs, trk


frames = simulate_scene(seed=SEED)
outputs, trk = run_tracker(frames)
n_det = sum(len(f["det_pos"]) for f in frames)
n_clutter = sum(sum(s == -1 for s in f["det_src"]) for f in frames)
occ_frames = sum(1 for f in frames if 0 in f["gt"] and not f["visible"][0])
ids_out = sorted({tid for o in outputs for tid in o})
print(f"{len(frames)} frames, {n_det} detections of which {n_clutter} clutter ({100 * n_clutter / n_det:.0f} %)")
print(f"Pedestrian occluded by the parked vehicle in {occ_frames} frames ({occ_frames * SCENE['dt']:.1f} s)")
print(f"Confirmed track IDs reported over the scene: {len(ids_out)}")

In [ ]:
def plot_scene(frames, outputs, ax=None, title="Scene: ground truth, detections and confirmed tracks"):
    ax = ax or plt.gca()
    (vx0, vx1), (vy0, vy1) = SCENE["van"]
    ax.add_patch(Rectangle((vx0, vy0), vx1 - vx0, vy1 - vy0, color="grey", alpha=0.5, label="parked vehicle (occluder)"))
    # detections
    D = [(p, s) for f in frames for p, s in zip(f["det_pos"], f["det_src"])]
    true_d = np.array([p for p, s in D if s >= 0]); clut = np.array([p for p, s in D if s < 0])
    if len(clut):
        ax.scatter(clut[:, 0], clut[:, 1], s=6, marker="x", color=OI["purple"], alpha=0.5, label="clutter detections")
    if len(true_d):
        ax.scatter(true_d[:, 0], true_d[:, 1], s=5, color="grey", alpha=0.6, label="object detections")
    # ground truth
    names = {0: "pedestrian", 1: "cyclist", 2: "car"}
    for oid in names:
        g = np.array([f["gt"][oid][0] for f in frames if oid in f["gt"]])
        ax.plot(g[:, 0], g[:, 1], color=OI["black"], lw=3, alpha=0.35)
        ax.annotate(names[oid], g[0], xytext=(4, 6), textcoords="offset points", fontsize=9)
    # tracks
    hist = {}
    for o in outputs:
        for tid, (x, P, c, miss) in o.items():
            hist.setdefault(tid, []).append(x[:2])
    palette = [OI["blue"], OI["orange"], OI["green"], OI["vermillion"], OI["skyblue"], OI["purple"], OI["yellow"]]
    for k, (tid, pts) in enumerate(sorted(hist.items())):
        pts = np.array(pts)
        ax.plot(pts[:, 0], pts[:, 1], "-", lw=1.8, color=palette[k % len(palette)])
        ax.annotate(f"#{tid}", pts[-1], fontsize=8, color=palette[k % len(palette)], fontweight="bold")
    ax.plot([], [], color=OI["black"], lw=3, alpha=0.35, label="ground truth")
    ax.plot([], [], color=OI["blue"], lw=1.8, label="confirmed tracks (#ID)")
    ax.scatter([0], [0], marker=">", s=150, color=OI["black"], label="ego sensor")
    ax.set(xlim=(-2, 62), ylim=(-15, 12), xlabel="x (longitudinal) [m]", ylabel="y (lateral) [m]", title=title)
    ax.set_aspect("equal"); ax.legend(fontsize=7, loc="upper left", ncol=2)
    return hist


plt.figure(figsize=(14, 6.5))
hist = plot_scene(frames, outputs)
plt.tight_layout(); plt.show()
lengths = {tid: len(p) for tid, p in hist.items()}
print("Text summary: confirmed tracks and their lengths (frames):",
      ", ".join(f"#{k}: {v}" for k, v in sorted(lengths.items())))

## 6. Measuring tracking quality

### 6.1 CLEAR MOT: MOTA, MOTP and identity switches

Bernardin and Stiefelhagen (2008) define, per frame $t$, a one-to-one matching between ground-truth objects and tracker hypotheses within a distance threshold, **preferring to keep last frame's correspondences** where still valid. With $\text{FN}_t$ misses, $\text{FP}_t$ false positives, $\text{IDSW}_t$ identity mismatches (a ground-truth object matched to a different track ID than before) and $\text{GT}_t$ ground-truth objects:

$$\text{MOTA} = 1 - \frac{\sum_t (\text{FN}_t + \text{FP}_t + \text{IDSW}_t)}{\sum_t \text{GT}_t}, \qquad
\text{MOTP} = \frac{\sum_{t,i} d_{t,i}}{\sum_t c_t},$$

where $d_{t,i}$ is the distance of matched pair $i$ and $c_t$ the number of matches. MOTA can be negative. It is dominated by detection errors (FN, FP are counted per frame, IDSW only once per switch), so it says relatively little about association quality.

### 6.2 HOTA

Luiten et al. (2021) proposed **HOTA** (Higher Order Tracking Accuracy) to balance detection and association. For a localisation threshold $\alpha$, after matching, each true positive $c$ is scored by how well its *whole* ground-truth trajectory and its *whole* predicted trajectory agree:

$$\mathcal{A}(c) = \frac{|\text{TPA}(c)|}{|\text{TPA}(c)| + |\text{FNA}(c)| + |\text{FPA}(c)|},\qquad
\text{DetA}_\alpha = \frac{|\text{TP}|}{|\text{TP}|+|\text{FN}|+|\text{FP}|},\qquad
\text{AssA}_\alpha = \frac{1}{|\text{TP}|}\sum_{c\in\text{TP}}\mathcal{A}(c),$$

$$\text{HOTA}_\alpha = \sqrt{\text{DetA}_\alpha\cdot\text{AssA}_\alpha}, \qquad \text{HOTA} = \frac{1}{19}\sum_{\alpha\in\{0.05,0.10,\dots,0.95\}}\text{HOTA}_\alpha .$$

Here TPA$(c)$ are the true positives sharing both the ground-truth ID and the predicted ID of $c$; FNA$(c)$ are the frames where that ground-truth object is matched to another ID or missed; FPA$(c)$ are the frames where that predicted ID is matched to another object or is a false positive. HOTA is the primary metric of the KITTI and MOTChallenge tracking benchmarks.

### 6.3 AMOTA

Weng et al. (2020) observed that MOTA depends on the detector's confidence threshold, and introduced **AMOTA**, the average of a (scaled) MOTA over a range of recall values obtained by sweeping that threshold, by analogy with average precision. AMOTA is the primary tracking metric of the nuScenes benchmark (Caesar et al., 2020). Our synthetic detector has no confidence scores, so we compute MOTA and HOTA only.

The implementation below uses point objects; similarity for HOTA is $S = \max(0, 1 - d/d_0)$ with $d_0 = 2$ m instead of box IoU, and matching per $\alpha$ uses the Hungarian algorithm on similarity alone (the reference implementation additionally rewards association in the matching score). Treat our numbers as *teaching approximations*; use the official TrackEval or nuScenes devkit code for reportable results.

In [ ]:
def to_eval_frames(frames, outputs):
    gts = [{i: g[0] for i, g in f["gt"].items()} for f in frames]
    hyps = [{tid: v[0][:2] for tid, v in o.items()} for o in outputs]
    return gts, hyps


def euclid(a, b):
    return float(np.linalg.norm(np.asarray(a)[:2] - np.asarray(b)[:2]))


def clear_mot(gts, hyps, thr=2.0, log=None, dist=euclid):
    # dist(gt_item, hyp_item): distance used for matching (default: Euclidean, in metres)
    # log: optional list that receives the per-frame correspondences {gt id: hypothesis id}
    last = {}                      # gt id -> last matched hypothesis id
    tp = fn = fp = idsw = 0; dist_sum = 0.0; n_gt = 0
    prev = {}
    for g, h in zip(gts, hyps):
        n_gt += len(g)
        match = {}
        for gi, hi in prev.items():              # keep valid correspondences
            if gi in g and hi in h and dist(g[gi], h[hi]) <= thr:
                match[gi] = hi
        rg = [i for i in g if i not in match]; rh = [j for j in h if j not in match.values()]
        if rg and rh:
            C = np.array([[dist(g[i], h[j]) for j in rh] for i in rg])
            C[C > thr] = BIG
            for a, b in zip(*linear_sum_assignment(C)):
                if C[a, b] < BIG:
                    match[rg[a]] = rh[b]
        for gi, hi in match.items():
            if gi in last and last[gi] != hi:
                idsw += 1
            last[gi] = hi
            dist_sum += dist(g[gi], h[hi])
        tp += len(match); fn += len(g) - len(match); fp += len(h) - len(match)
        prev = match
        if log is not None:
            log.append(dict(match))
    return dict(MOTA=1 - (fn + fp + idsw) / max(n_gt, 1), MOTP=dist_sum / max(tp, 1),
                IDSW=idsw, FN=fn, FP=fp, GT=n_gt)


def hota(gts, hyps, d0=2.0, alphas=np.arange(0.05, 0.96, 0.05), dist=euclid):
    # Matching uses the similarity S = max(0, 1 - d/d0) alone, so one Hungarian solution per frame serves every alpha
    gt_count, hyp_count, frames_m = {}, {}, []
    for g, h in zip(gts, hyps):
        for i in g: gt_count[i] = gt_count.get(i, 0) + 1
        for j in h: hyp_count[j] = hyp_count.get(j, 0) + 1
        gi, hj = list(g), list(h)
        cand = []
        if gi and hj:
            Sim = np.maximum(0.0, 1 - np.array([[dist(g[i], h[j]) for j in hj] for i in gi]) / d0)
            cand = [(gi[r_], hj[c_], Sim[r_, c_]) for r_, c_ in zip(*linear_sum_assignment(-Sim))]
        frames_m.append((len(gi), len(hj), cand))
    res = []
    for a in alphas:
        pairs = []                                 # list of (gt_id, hyp_id) true positives
        n_fn = n_fp = 0
        for n_g, n_h, cand in frames_m:
            m = 0
            for i, j, s in cand:
                if s >= a and s > 0:
                    pairs.append((i, j)); m += 1
            n_fn += n_g - m; n_fp += n_h - m
        n_tp = len(pairs)
        if n_tp == 0:
            res.append((0.0, 0.0, 0.0)); continue
        pair_count = {}
        for p in pairs: pair_count[p] = pair_count.get(p, 0) + 1
        ass = sum(pair_count[(i, j)] / (gt_count[i] + hyp_count[j] - pair_count[(i, j)]) for i, j in pairs) / n_tp
        det = n_tp / (n_tp + n_fn + n_fp)
        res.append((np.sqrt(det * ass), det, ass))
    res = np.array(res)
    return dict(HOTA=res[:, 0].mean(), DetA=res[:, 1].mean(), AssA=res[:, 2].mean())


def time_to_confirm(frames, outputs, oid, thr=2.0, after=0.0):
    # Delay [s] between the object first being visible (after time `after`) and a confirmed track lying within thr
    t_vis = next((f["t"] for f in frames if f["t"] >= after and oid in f["gt"] and f["visible"][oid]), None)
    if t_vis is None:
        return np.nan
    for f, o in zip(frames, outputs):
        if f["t"] >= t_vis and oid in f["gt"]:
            if any(np.linalg.norm(v[0][:2] - f["gt"][oid][0]) <= thr for v in o.values()):
                return f["t"] - t_vis
    return np.inf


gts, hyps = to_eval_frames(frames, outputs)
m = clear_mot(gts, hyps); h = hota(gts, hyps)
print("CLEAR MOT:", {k: (round(v, 3) if isinstance(v, float) else v) for k, v in m.items()})
print("HOTA (point-object approximation):", {k: round(v, 3) for k, v in h.items()})
print(f"Pedestrian time-to-confirm: {time_to_confirm(frames, outputs, 0):.1f} s after first visibility")

### 6.4 Coasting through the occlusion

With the default $K_{\max} = 8$ frames (0.8 s), the pedestrian's track dies behind the parked vehicle and a new track has to be born and confirmed when she reappears. The planner briefly loses the object, and the new track starts **without velocity**. Let us compare several coasting limits, averaging over several random seeds.

In [ ]:
# time at which the pedestrian becomes visible again after the occlusion (geometry is seed-independent)
ped_vis = [(f["t"], f["visible"][0]) for f in frames if 0 in f["gt"]]
t_occ_start = next(t for (t, v), (_, v_prev) in zip(ped_vis[1:], ped_vis[:-1]) if v_prev and not v)
t_reappear = next(t for t, v in ped_vis if t > t_occ_start and v)
print(f"Pedestrian occluded from t = {t_occ_start:.1f} s to t = {t_reappear:.1f} s")

COAST_SEEDS = range(300, 306)
rows = []
for k_max in [3, 8, 15, 25, 40]:
    for s in COAST_SEEDS:
        fr = simulate_scene(seed=s)
        out_k, _ = run_tracker(fr, max_coast=k_max)
        g_, h_ = to_eval_frames(fr, out_k)
        mk, hk = clear_mot(g_, h_), hota(g_, h_)
        ped_ids = {tid for f, o in zip(fr, out_k) if 0 in f["gt"]
                   for tid, v in o.items() if np.linalg.norm(v[0][:2] - f["gt"][0][0]) <= 2.0}
        rows.append({"K_max [frames]": k_max, "coast limit [s]": k_max * SCENE["dt"],
                     "pedestrian track IDs": len(ped_ids), "IDSW": mk["IDSW"], "FP frames": mk["FP"],
                     "MOTA": mk["MOTA"], "HOTA": hk["HOTA"], "AssA": hk["AssA"]})
coast_df = pd.DataFrame(rows).groupby("K_max [frames]").mean()
print(f"Mean over {len(COAST_SEEDS)} seeds:")
print(coast_df.round(2).to_string())
best_hota = coast_df["HOTA"].idxmax()
print(f"\nText summary: the pedestrian is carried on average by {coast_df['pedestrian track IDs'].iloc[0]:.2f} track IDs at "
      f"K_max={coast_df.index[0]} and {coast_df['pedestrian track IDs'].iloc[-1]:.2f} at K_max={coast_df.index[-1]}; "
      f"false-positive frames change from {coast_df['FP frames'].iloc[0]:.1f} to {coast_df['FP frames'].iloc[-1]:.1f}. "
      f"HOTA is highest at K_max={best_hota}, MOTA at K_max={coast_df['MOTA'].idxmax()}.")

Longer coasting keeps the pedestrian's identity (fewer IDs, better AssA) but every confirmed clutter track also coasts longer before it is deleted (more false-positive frames). Note that MOTA and HOTA need not agree on the best setting. There is no free lunch: the coasting limit must be argued from the occlusion durations expected in the ODD and from the cost of a ghost object for the downstream planner.

## 7. Discarding history on reclassification: a simplified analogue

We now give the pedestrian's detections noisy class labels: with probability $p_{\text{flip}}$ each detection is labelled as one of the *other* classes (cyclist, vehicle, other). We compare

- **retain history** (default): association is class-agnostic; the track keeps its state and takes a majority vote over class labels;
- **reset on reclassification** (`reset_on_reclass=True`): a label change discards the track, and a new tentative track (zero velocity, no history) is started.

For each frame in which the pedestrian is visible, we ask: *does planning receive a confirmed track for her, and how wrong is its 2-second-ahead constant-velocity prediction?*

In [ ]:
def ped_prediction_quality(frames, outputs, horizon=2.0, thr=2.0):
    covered, errs, vel_err = 0, [], []
    n_vis = 0
    for f, o in zip(frames, outputs):
        if 0 not in f["gt"] or not f["visible"][0] or f["t"] + horizon > SCENE["T"]:
            continue
        n_vis += 1
        p_true = f["gt"][0][0]
        cand = [v for v in o.values() if np.linalg.norm(v[0][:2] - p_true) <= thr]
        if cand:
            x = min(cand, key=lambda v: np.linalg.norm(v[0][:2] - p_true))[0]
            covered += 1
            future_true = ground_truth(f["t"] + horizon)[0][0]
            errs.append(np.linalg.norm(x[:2] + horizon * x[2:] - future_true))
            vel_err.append(np.linalg.norm(x[2:] - f["gt"][0][1]))
    return covered / max(n_vis, 1), np.array(errs), np.array(vel_err)


p_flips = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
n_seeds = 6
rows = []
for pf in p_flips:
    for s in range(n_seeds):
        fr = simulate_scene(p_d=0.9, p_mislabel={"pedestrian": pf, "cyclist": 0.05, "vehicle": 0.05, "other": 0.05},
                            seed=100 + s)
        for mode in [False, True]:
            out, trk_ = run_tracker(fr, reset_on_reclass=mode)
            cov, errs, verr = ped_prediction_quality(fr, out)
            rows.append({"p_flip": pf, "seed": s, "mode": "reset on reclass" if mode else "retain history",
                         "coverage": cov, "pred_err_2s_m": np.median(errs) if len(errs) else np.nan,
                         "resets": trk_.n_reclass_resets})
recl = pd.DataFrame(rows)
summary = recl.groupby(["p_flip", "mode"])[["coverage", "pred_err_2s_m", "resets"]].mean().unstack("mode")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for mode, c in [("retain history", OI["blue"]), ("reset on reclass", OI["vermillion"])]:
    axes[0].plot(p_flips, 100 * summary[("coverage", mode)], "o-", color=c, label=mode)
    axes[1].plot(p_flips, summary[("pred_err_2s_m", mode)], "o-", color=c, label=mode)
axes[0].set(title="Visible frames with a confirmed pedestrian track", xlabel="label-flip probability p_flip [-]",
            ylabel="coverage [%]", ylim=(0, 105))
axes[1].set(title="Median 2 s-ahead prediction error (when tracked)", xlabel="label-flip probability p_flip [-]",
            ylabel="position error after 2 s [m]")
for a in axes: a.legend()
plt.tight_layout(); plt.show()

print(summary.round(2).to_string())
r_hi = summary.loc[0.3]
print(f"\nText summary (p_flip = 0.3, mean of {n_seeds} seeds): coverage {100 * r_hi[('coverage', 'retain history')]:.0f} % "
      f"when history is retained vs {100 * r_hi[('coverage', 'reset on reclass')]:.0f} % when it is reset; "
      f"median 2 s prediction error {r_hi[('pred_err_2s_m', 'retain history')]:.2f} m vs "
      f"{r_hi[('pred_err_2s_m', 'reset on reclass')]:.2f} m.")

Two effects compound when history is discarded. First, every reset sends the object back to the *tentative* state, so with frequent label changes it is rarely confirmed at all. Second, when a fresh track is confirmed it has had only a few frames to learn the velocity, so its prediction is poor. Classification uncertainty therefore turns into **motion-prediction** uncertainty. The engineering lesson is general: **object identity and kinematic history should be robust to classification changes**, and class uncertainty should be propagated (e.g. as a class distribution, or by planning against the most conservative plausible class), not used as a reason to forget an object.

## 8. Experiment: detection dropout and noise as triggering conditions

ISO 21448:2022 asks for the identification of **triggering conditions**: specific conditions of a scenario that initiate a reaction of the system contributing to hazardous behaviour, or to an inability to prevent or mitigate it (see [Module 04, Notebook 02](../04_SOTIF_ISO21448/02_triggering_conditions_and_scenarios.ipynb)). Many environmental triggering conditions act on a tracker through just two channels: they lower the per-frame **detection probability** $P_D$ (heavy rain, glare, partial occlusion, an unusual object appearance) and they increase the **measurement noise** $\sigma_z$ (spray, multipath, poor calibration).

We sweep both over a grid, with several random seeds per cell, and record the pedestrian's **time-to-confirm**, the **number of distinct track IDs** that carry her (fragmentation; 2 is the minimum here because the default coasting limit does not bridge the occlusion) and **MOTA**. We run the tracker in two configurations:

- **noise model matched**: the tracker's $\mathbf{R}$ is set to the true $\sigma_z$ (the degradation is known, e.g. reported by sensor self-diagnosis);
- **noise model fixed**: $\mathbf{R}$ stays at the design value $\sigma_z = 0.3$ m, the situation of a deployed system meeting an unforeseen condition.

> **Illustrative data:** values below are synthetic, chosen for teaching; they are not measurements.

In [ ]:
pd_grid = [0.95, 0.85, 0.75, 0.65, 0.55, 0.45]
sz_grid = [0.15, 0.3, 0.6, 1.0]
N_SEEDS = 4
t_start = time.time()
recs = []
for p_d in pd_grid:
    for sz in sz_grid:
        for s in range(N_SEEDS):
            fr = simulate_scene(p_d=p_d, sigma_z=sz, seed=1000 + s)
            for mode, r_sigma in [("matched", sz), ("fixed", 0.3)]:
                out, _ = run_tracker(fr, sigma_z=r_sigma)
                g_, h_ = to_eval_frames(fr, out)
                mm = clear_mot(g_, h_)
                ped_ids = {tid for f, o in zip(fr, out) if 0 in f["gt"]
                           for tid, v in o.items() if np.linalg.norm(v[0][:2] - f["gt"][0][0]) <= 2.0}
                ttc = time_to_confirm(fr, out, 0)
                recs.append({"P_D": p_d, "sigma_z": sz, "noise model": mode, "seed": s, "MOTA": mm["MOTA"],
                             "IDSW": mm["IDSW"], "ped_track_ids": len(ped_ids),
                             "ttc_ped": ttc if np.isfinite(ttc) else SCENE["T"]})   # never confirmed -> scene length
sweep = pd.DataFrame(recs)
print(f"{len(sweep)} tracker runs in {time.time() - t_start:.1f} s")

metrics = [("ttc_ped", "Pedestrian time-to-confirm [s]", "{:.2f}"),
           ("ped_track_ids", "Pedestrian track IDs [-]", "{:.1f}"),
           ("MOTA", "MOTA [-]", "{:.2f}")]
piv = {(mode, col): sweep[sweep["noise model"] == mode].pivot_table(index="P_D", columns="sigma_z", values=col,
                                                                    aggfunc="mean").sort_index(ascending=False)
       for mode in ["matched", "fixed"] for col, _, _ in metrics}

fig, axes = plt.subplots(2, 3, figsize=(17, 9))
for c, (col, title, fmt) in enumerate(metrics):
    vals = np.concatenate([piv[(m, col)].values.ravel() for m in ["matched", "fixed"]])
    vmin, vmax = vals.min(), vals.max()
    cmap = "viridis_r" if col == "MOTA" else "viridis"          # bright = worse in every panel
    for r_, mode in enumerate(["matched", "fixed"]):
        ax, P = axes[r_, c], piv[(mode, col)]
        im = ax.imshow(P.values, cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")
        ax.set_xticks(range(len(P.columns))); ax.set_xticklabels([f"{x:g}" for x in P.columns])
        ax.set_yticks(range(len(P.index))); ax.set_yticklabels([f"{x:g}" for x in P.index])
        ax.set(xlabel="true measurement noise sigma_z [m]", ylabel="detection probability P_D [-]",
               title=f"{title}\nnoise model {mode}" + (" (R = true noise)" if mode == "matched" else " (R at 0.3 m)"))
        ax.grid(False)
        for (i, j), v in np.ndenumerate(P.values):
            frac = (v - vmin) / (vmax - vmin + 1e-12)
            dark_cell = frac > 0.5 if col == "MOTA" else frac < 0.5
            ax.text(j, i, fmt.format(v), ha="center", va="center", fontsize=8,
                    color="white" if dark_cell else "black")
        plt.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle("Effect of detection dropout and noise on track continuity (mean of seeds; brighter = worse)",
             fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
v50 = 50 / 3.6
tm, tf = piv[("matched", "ttc_ped")], piv[("fixed", "ttc_ped")]
fm, ff = piv[("matched", "ped_track_ids")], piv[("fixed", "ped_track_ids")]
mm_, mf = piv[("matched", "MOTA")], piv[("fixed", "MOTA")]
print("Text summary of the sweep:")
print(f"  - Lowering P_D from 0.95 to 0.45 (sigma_z = 0.3 m) raises the mean pedestrian time-to-confirm from "
      f"{tm.loc[0.95, 0.3]:.2f} s to {tm.loc[0.45, 0.3]:.2f} s, i.e. "
      f"{(tm.loc[0.45, 0.3] - tm.loc[0.95, 0.3]) * v50:.1f} m of additional travel at 50 km/h before the object is reported.")
print(f"  - With the noise model matched, MOTA stays between {mm_.values.min():.2f} and {mm_.values.max():.2f} "
      f"across the grid; coasting hides many missed detections from a per-frame metric.")
print(f"  - With the noise model fixed at 0.3 m and true sigma_z = 1.0 m (P_D = 0.95), the pedestrian is carried by "
      f"{ff.loc[0.95, 1.0]:.1f} track IDs on average (matched: {fm.loc[0.95, 1.0]:.1f}) and MOTA drops to "
      f"{mf.loc[0.95, 1.0]:.2f} (matched: {mm_.loc[0.95, 1.0]:.2f}).")
worse = "fixed" if (mf.values.mean() < mm_.values.mean()) else "matched"
print(f"  - Averaged over the grid, the '{worse}' configuration has the lower MOTA "
      f"({mf.values.mean():.2f} fixed vs {mm_.values.mean():.2f} matched): an unmodelled noise increase is more "
      f"damaging than a known one.")

**Interpretation for a SOTIF analysis.**

- The tracker converts a *per-frame* performance limitation (lower $P_D$) into a *temporal* one (later confirmation, broken identities). Acceptance criteria and validation targets should therefore be stated at the level that matters to the hazard, for example *"a crossing pedestrian is reported as a confirmed track with velocity within 0.5 s of becoming visible, in at least X % of cases"*, not only as per-frame detection recall.
- Noise beyond the tuned value causes gate misses (the filter is now over-confident, Section 2), which show up as fragmentation and identity switches even when $P_D$ is high. This is a functional insufficiency of the *tracker specification*, triggered by a sensor-level condition.
- Each axis of the sweep corresponds to identifiable triggering conditions. The grid is a miniature of the systematic, scenario-based exploration discussed in [Module 04, Notebook 04](../04_SOTIF_ISO21448/04_simulation_based_validation.ipynb).
- An online NIS monitor (Section 2.4) can detect some of these conditions at runtime, an idea developed in [Module 07](../07_Runtime_Assurance_and_Operations/01_odd_and_runtime_monitoring.ipynb).

## 9. Real data: tracking pedestrians and cars in a KITTI sequence

Everything so far ran on a simulated scene, where we controlled $P_D$, noise and occlusion. We now apply the same filter, association and track management to a real recording: **KITTI tracking sequence 0015** (Geiger et al., 2012; 2013), an urban drive in Karlsruhe in which the test vehicle brakes to a stop at a pedestrian crossing while pedestrians and cyclists cross in front of it and occlude one another. Every labelled object carries a **track ID** that persists through occlusions, which is exactly the reference that MOTA, IDSW and HOTA need.

The section has three parts:

1. a **real detector** (torchvision Faster R-CNN with a MobileNetV3 backbone, pretrained on COCO) on 40 camera frames, which gives measured detection probabilities by occlusion level;
2. **tracking those detections in the image plane** with the tracker of Section 4, scored against the real track IDs, including the identity switches it makes on real frames;
3. a **bird's-eye-view (BEV) run over the full 37.6 s sequence**, where detections are generated from the labelled 3D positions and degraded with the detection rates measured in part 1.

> **Data and licence.** KITTI is licensed CC BY-NC-SA 3.0 (non-commercial); cite Geiger et al. (2012, 2013). Frames are fetched individually and cached; the first run downloads about 40 images (roughly a minute), later runs read the cache.

In [ ]:
# Course toolkit (tables, formulas and real-data loaders). Installs automatically on Colab.
try:
    import avsafety
except ImportError:
    %pip install -q "avsafety[data] @ git+https://github.com/milinpatel07/Autonomous-Driving_AI-Safety-and-Security.git"
    import avsafety
from avsafety import data as D

SEQ = 15
IMG_FRAMES = list(range(50, 90))     # 4.0 s: ego brakes to a stop, pedestrians and cyclists cross and occlude each other
CLASS_MAP = {"Car": "vehicle", "Van": "vehicle", "Truck": "vehicle", "Pedestrian": "pedestrian", "Cyclist": "cyclist"}
IGNORE_TYPES = ["Person_sitting", "Misc", "Tram", "DontCare"]      # neither required nor counted as false positive

t_load = time.time()
try:
    kt = D.kitti_tracking(SEQ, frames=IMG_FRAMES, images=True)
    calib_txt = D.fetch_kitti_file("data_tracking_calib.zip", f"training/calib/{SEQ:04d}.txt").decode()
except D.DataUnavailable as exc:
    raise RuntimeError(f"KITTI could not be downloaded ({exc}); Sections 1-8 do not need it.") from None
labels_all = kt["labels"]
lab = labels_all[labels_all["type"].isin(CLASS_MAP)].copy()
lab["cls"] = lab["type"].map(CLASS_MAP)
lab["range_m"] = np.hypot(lab["x"], lab["z"])
n_frames = int(labels_all["frame"].max()) + 1
print(f"KITTI tracking sequence {SEQ:04d}: {n_frames} frames at 10 Hz ({n_frames / 10:.1f} s); "
      f"{len(IMG_FRAMES)} camera images loaded in {time.time() - t_load:.1f} s")
print(lab.groupby("cls")["track_id"].agg(tracks="nunique", labelled_boxes="size").to_string())
print(D.attribution("KITTI"))

### 9.1 A world frame from GPS/IMU

KITTI labels are given in the camera frame of the *moving* vehicle. Tracking in that frame mixes object motion with ego motion: a parked car appears to approach at the ego speed, and a constant-velocity model is violated whenever the ego vehicle brakes or turns. We therefore transform every position into a fixed **world frame** using the GPS/IMU (OXTS) poses, following the KITTI development kit: Mercator projection of latitude/longitude, roll–pitch–yaw rotation, and the calibrated IMU–LiDAR–camera chain. A simple check of the result: parked vehicles must stand still in the world frame.

In [ ]:
OXTS = kt["oxts"]                    # one GPS/IMU record per frame


def oxts_poses(oxts):
    # IMU-to-world poses (4x4) as in the KITTI devkit (Mercator projection), relative to the first frame
    er = 6378137.0
    scale = np.cos(np.radians(oxts["lat"].iloc[0]))
    poses = []
    for r in oxts.itertuples():
        t = np.array([scale * np.radians(r.lon) * er,
                      scale * er * np.log(np.tan(np.radians(90.0 + r.lat) / 2.0)), r.alt])
        cr, sr, cp, sp = np.cos(r.roll), np.sin(r.roll), np.cos(r.pitch), np.sin(r.pitch)
        cy, sy = np.cos(r.yaw), np.sin(r.yaw)
        Rx = np.array([[1, 0, 0], [0, cr, -sr], [0, sr, cr]])
        Ry = np.array([[cp, 0, sp], [0, 1, 0], [-sp, 0, cp]])
        Rz = np.array([[cy, -sy, 0], [sy, cy, 0], [0, 0, 1]])
        T = np.eye(4); T[:3, :3] = Rz @ Ry @ Rx; T[:3, 3] = t
        poses.append(T)
    T0inv = np.linalg.inv(poses[0])
    return np.array([T0inv @ T for T in poses])


def calib_matrix(text, key):
    vals = next(l for l in text.splitlines() if l.startswith(key)).split()[1:]
    T = np.eye(4); T[:3, :4] = np.array(vals, dtype=float).reshape(3, 4)
    return T


POSES = oxts_poses(OXTS)
C = kt["calib"]
RECT_TO_IMU = (np.linalg.inv(calib_matrix(calib_txt, "Tr_imu_velo")) @ np.linalg.inv(C["Tr_velo_to_cam"])
               @ np.linalg.inv(C["R0"]))


def rect_to_world(pts_rect, frame):
    # rectified camera coordinates (N x 3) at a frame -> world ground plane (N x 2) [m]
    pts = np.atleast_2d(pts_rect)
    homo = np.hstack([pts, np.ones((len(pts), 1))])
    return (POSES[frame] @ RECT_TO_IMU @ homo.T).T[:, :2]


lab[["wx", "wy"]] = np.nan
for f, idx in lab.groupby("frame").groups.items():
    lab.loc[idx, ["wx", "wy"]] = rect_to_world(lab.loc[idx, ["x", "y", "z"]].values, f)
speed = np.hypot(OXTS["vf"], OXTS["vl"]).values
spread = lab.groupby("track_id").agg(cls=("cls", "first"), frames=("frame", "size"),
                                     sd_x_world=("wx", "std"), sd_y_world=("wy", "std"),
                                     z_range_camera=("z", lambda z: z.max() - z.min()))
parked = spread[spread["frames"] > 100].sort_values("sd_x_world")
print(f"Ego speed {speed[0]:.1f} m/s at frame 0, {speed[IMG_FRAMES[0]]:.1f} m/s at frame {IMG_FRAMES[0]}, "
      f"{speed[IMG_FRAMES[-1]]:.1f} m/s at frame {IMG_FRAMES[-1]}; {np.sum(speed) * 0.1:.0f} m driven in the sequence.")
print("Objects labelled in more than 100 frames: change of distance in the camera frame vs. spread in the world frame [m]")
print(parked.round(2).to_string())
static = parked[parked[["sd_x_world", "sd_y_world"]].max(axis=1) < 0.5]
p = static.loc[static["z_range_camera"].idxmax()]
print(f"Text summary: object {p.name} ({p['cls']}) approaches the camera by {p['z_range_camera']:.0f} m in the camera "
      f"frame but stays within {max(p['sd_x_world'], p['sd_y_world']):.2f} m (SD) in the world frame: it is stationary, "
      f"and the ego-motion compensation works.")

### 9.2 A real detector on real frames

We run the pretrained detector on the 40 frames and match its boxes to the labelled boxes of the same frame (Hungarian assignment, IoU ≥ 0.5). One adaptation is needed: KITTI's *Cyclist* box encloses rider and bicycle, whereas a COCO *person* box covers the rider only, so for cyclists we also accept a detection that lies at least 70 % inside the labelled box. Unmatched detections inside *DontCare* or ignored regions are not counted as false positives, as in the KITTI evaluation.

The result is a **measured** per-frame detection probability $P_D$ for each class and KITTI occlusion level (0 fully visible, 1 partly occluded, 2 largely occluded, 3 unknown). These are the numbers that Section 4.1 and Section 8 treated as free parameters.

In [ ]:
import torch
import torchvision

torch.set_num_threads(1)     # about 1 s per frame on one CPU thread; more threads stall on a busy or shared CPU
torch.manual_seed(SEED)
COCO_TO_CLS = {1: "pedestrian", 3: "vehicle", 6: "vehicle", 8: "vehicle"}     # COCO person, car, bus, truck
SCORE_THR = 0.5

t_det = time.time()
weights = torchvision.models.detection.FasterRCNN_MobileNet_V3_Large_FPN_Weights.DEFAULT
detector = torchvision.models.detection.fasterrcnn_mobilenet_v3_large_fpn(weights=weights).eval()
dets_img = {}
with torch.no_grad():
    for f, img in zip(IMG_FRAMES, kt["images"]):
        out = detector([torch.from_numpy(img.copy()).permute(2, 0, 1).float() / 255.0])[0]
        dets_img[f] = [(b, int(l), float(s)) for b, l, s in
                       zip(out["boxes"].numpy(), out["labels"].numpy(), out["scores"].numpy())
                       if int(l) in COCO_TO_CLS and s >= SCORE_THR]
print(f"Detector: {len(IMG_FRAMES)} frames in {time.time() - t_det:.1f} s, "
      f"{sum(len(v) for v in dets_img.values())} person/vehicle detections with score >= {SCORE_THR}")


def iou(a, b):
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    return inter / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter + 1e-9)


def ioa(box, region):
    # fraction of `box` lying inside `region`
    ix = max(0.0, min(box[2], region[2]) - max(box[0], region[0])); iy = max(0.0, min(box[3], region[3]) - max(box[1], region[1]))
    return ix * iy / ((box[2] - box[0]) * (box[3] - box[1]) + 1e-9)


def match_quality(g_box, g_cls, d_box):
    q = iou(g_box, d_box)
    if g_cls == "cyclist" and ioa(d_box, g_box) >= 0.7:
        q = max(q, 0.5)
    return q


ignore_lab = labels_all[labels_all["type"].isin(IGNORE_TYPES)]
rows, fp_rows = [], []
for f in IMG_FRAMES:
    G = lab[lab["frame"] == f]
    db = [d[0] for d in dets_img[f]]
    matched_g, matched_d = {}, set()
    if len(G) and len(db):
        Mq = np.array([[match_quality(g, c, d) for d in db] for g, c in zip(G[["x1", "y1", "x2", "y2"]].values, G["cls"])])
        for a, b in zip(*linear_sum_assignment(-Mq)):
            if Mq[a, b] >= 0.5:
                matched_g[a] = b; matched_d.add(b)
    for a, g in enumerate(G.itertuples()):
        rows.append({"frame": f, "track_id": g.track_id, "cls": g.cls, "occluded": g.occluded,
                     "height_px": g.y2 - g.y1, "range_m": g.range_m, "detected": a in matched_g,
                     "det": matched_g.get(a, -1)})
    ign = ignore_lab[ignore_lab["frame"] == f][["x1", "y1", "x2", "y2"]].values
    for b, d in enumerate(dets_img[f]):
        if b not in matched_d and not any(ioa(d[0], r) > 0.5 for r in ign):
            fp_rows.append({"frame": f, "cls": COCO_TO_CLS[d[1]], "score": d[2]})
det_eval = pd.DataFrame(rows)
fp_df = pd.DataFrame(fp_rows)
FP_PER_FRAME = len(fp_df) / len(IMG_FRAMES)
pd_tab = det_eval.pivot_table(index="occluded", columns="cls", values="detected", aggfunc=["mean", "size"])
pd_tab.index = pd_tab.index.map({0: "0 fully visible", 1: "1 partly occluded", 2: "2 largely occluded", 3: "3 unknown"})
print("Measured per-frame detection probability P_D (mean) and number of labelled boxes (size):")
print(pd_tab.round(2).to_string())
print(f"False positives outside ignore regions: {len(fp_df)} in {len(IMG_FRAMES)} frames ({FP_PER_FRAME:.2f} per frame)")
ped = det_eval[det_eval["cls"] == "pedestrian"]
p0 = ped.loc[ped["occluded"] == 0, "detected"].mean(); p2 = ped.loc[ped["occluded"] == 2, "detected"].mean()
print(f"Text summary: a fully visible pedestrian is detected in {100 * p0:.0f} % of frames, a largely occluded one in "
      f"{100 * p2:.0f} %. In the terms of Section 4.1, occlusion moves P_D from the comfortable to the failing end of the curve.")

In [ ]:
from matplotlib.patches import Rectangle as Rect
SHOW = [IMG_FRAMES[0], IMG_FRAMES[20], IMG_FRAMES[-1]]
CLS_COL = {"pedestrian": OI["vermillion"], "cyclist": OI["orange"], "vehicle": OI["blue"]}
fig, axes = plt.subplots(len(SHOW), 1, figsize=(15, 4.6 * len(SHOW)))
for ax, f in zip(axes, SHOW):
    ax.imshow(kt["images"][IMG_FRAMES.index(f)]); ax.grid(False); ax.set_xticks([]); ax.set_yticks([])
    for r in det_eval[det_eval["frame"] == f].itertuples():
        g = lab[(lab["frame"] == f) & (lab["track_id"] == r.track_id)].iloc[0]
        ax.add_patch(Rect((g.x1, g.y1), g.x2 - g.x1, g.y2 - g.y1, fill=False, lw=2, color=CLS_COL[r.cls],
                          ls="-" if r.detected else ":"))
        ax.text(g.x1, g.y1 - 3, f"{r.track_id} occ{r.occluded}", color=CLS_COL[r.cls], fontsize=8, fontweight="bold")
    for d in dets_img[f]:
        b = d[0]
        ax.add_patch(Rect((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, lw=1, color=OI["green"]))
    ax.set_title(f"Frame {f}: labelled objects (solid = detected, dotted = missed; 'occ' = KITTI occlusion level), "
                 f"detector boxes in green", fontsize=10)
plt.tight_layout(); plt.show()

### 9.3 Why we do not lift these boxes into 3D

A tempting shortcut is to place each 2D box on a flat road: with the camera 1.65 m above the ground (Geiger et al., 2013), the image row of the box's lower edge gives a range. The table below compares that estimate with the labelled 3D positions for every matched detection. Within 10 m the error is about a metre, at 15–20 m a few metres, and beyond 20 m it reaches tens of metres, because one pixel row then spans several metres of road, the vehicle pitches while braking, and an occluded lower edge is placed too high. A tracker fed with such ranges would mostly measure the ranging error. Real systems obtain range from LiDAR, radar or stereo, or from learned monocular depth with its own validated error model. We therefore track the detections **in the image** (Section 9.4) and use the labelled 3D positions, with degradations measured here, for the BEV analysis (Section 9.5).

In [ ]:
CAM_H = 1.65
P2 = C["P2"]


def box_to_ground(box):
    # bottom-centre pixel of a 2D box -> rectified camera point on a flat road at height CAM_H below the camera
    u, v = 0.5 * (box[0] + box[2]), box[3]
    fx, cx, fy, cy = P2[0, 0], P2[0, 2], P2[1, 1], P2[1, 2]
    if v <= cy + 2:
        return None                                    # at or above the horizon
    Z = (fy * CAM_H + P2[1, 3] - v * P2[2, 3]) / (v - cy)
    return np.array([(u * (Z + P2[2, 3]) - cx * Z - P2[0, 3]) / fx, CAM_H, Z])


err = []
for r in det_eval[det_eval["detected"]].itertuples():
    g = lab[(lab["frame"] == r.frame) & (lab["track_id"] == r.track_id)].iloc[0]
    p = box_to_ground(dets_img[r.frame][r.det][0])
    if p is not None:
        err.append({"range_m": g.range_m, "err_depth": p[2] - g.z, "err_lateral": p[0] - g.x})
err = pd.DataFrame(err)
robust_sd = lambda e: 1.4826 * np.median(np.abs(e - np.median(e)))      # SD estimate that ignores outliers
gp_tab = err.groupby(pd.cut(err["range_m"], [0, 10, 15, 20, 30, 60]), observed=True).agg(
    n=("err_depth", "size"), median_depth_error_m=("err_depth", "median"), robust_sd_depth_m=("err_depth", robust_sd),
    robust_sd_lateral_m=("err_lateral", robust_sd))
gp_tab.index = gp_tab.index.astype(str)
print("Flat-road range from the lower box edge vs labelled 3D position (matched detections):")
print(gp_tab.round(2).to_string())

### 9.4 Tracking the real detections in the image plane

The tracker of Section 4 does not care about units. We give it the **box centre in pixels** as the measurement, a measurement noise of a few pixels, and a process noise large enough for image-plane motion (a crossing pedestrian at 10 m moves about 100 px/s; braking pitches the whole image). This is essentially SORT (Bewley et al., 2016) with Mahalanobis gating instead of IoU. The confirmation and coasting parameters stay at $M=3$ of $N=5$ and $K_{\max}=8$. The two noise values ($\sigma_a = 200$ px/s², $\sigma_z = 8$ px) were picked from a small grid ($\sigma_a \in \{100, 200, 400\}$, $\sigma_z \in \{4, 8, 15\}$) on this same window, so the scores below are slightly optimistic; a real tuning would use separate sequences.

For scoring we compare track positions with the centres of the labelled boxes and use a **scale-aware distance**: the centre distance divided by the labelled box height. A match requires a distance below 0.5 (half a box height) for CLEAR MOT; HOTA uses the similarity $S=\max(0, 1-d)$. Labelled objects smaller than 25 px or more than 50 % truncated are *ignore* objects, as are the ignore types above: a track on them is neither a hit nor a false positive.

In [ ]:
IMG_CFG = dict(sigma_a=200.0, sigma_z=8.0, v0_var=150.0**2, M=3, N=5, max_coast=8)     # pixels, pixels/s^2


def scaled_dist(g, h):
    # centre distance in units of the labelled box height
    return float(np.linalg.norm(g[:2] - h[:2]) / g[2])


def image_ground_truth(f):
    G = lab[lab["frame"] == f]
    ok = ((G["y2"] - G["y1"]) >= 25) & (G["truncated"] <= 0.5)
    ev = {int(r.track_id): np.array([(r.x1 + r.x2) / 2, (r.y1 + r.y2) / 2, r.y2 - r.y1]) for r in G[ok].itertuples()}
    ign = [np.array([r.x1, r.y1, r.x2, r.y2]) for r in G[~ok].itertuples()]
    ign += [np.array([r.x1, r.y1, r.x2, r.y2]) for r in ignore_lab[ignore_lab["frame"] == f].itertuples()]
    return ev, ign


def drop_on_ignored(hyp, gt, ign_boxes):
    # remove hypotheses that are not near any evaluated object but lie inside an ignore box
    out = {}
    for k, v in hyp.items():
        near_gt = any(scaled_dist(g, v) <= 0.5 for g in gt.values())
        in_ign = any(b[0] <= v[0] <= b[2] and b[1] <= v[1] <= b[3] for b in ign_boxes)
        if near_gt or not in_ign:
            out[k] = v
    return out


Track._next_id = 0
img_trk = MultiObjectTracker(dt=0.1, fov=None, **IMG_CFG)
out_img, gts_i, hyps_i = [], [], []
for f in IMG_FRAMES:
    centres = [np.array([(b[0] + b[2]) / 2, (b[1] + b[3]) / 2]) for b, _, _ in dets_img[f]]
    out_img.append(img_trk.step(centres, [COCO_TO_CLS[l] for _, l, _ in dets_img[f]]))
    ev, ign = image_ground_truth(f)
    gts_i.append(ev)
    hyps_i.append(drop_on_ignored({k: v[0][:2] for k, v in out_img[-1].items()}, ev, ign))
log_i = []
m_img = clear_mot(gts_i, hyps_i, thr=0.5, dist=scaled_dist, log=log_i)
h_img = hota(gts_i, hyps_i, d0=1.0, dist=scaled_dist)
print(f"Real detections, frames {IMG_FRAMES[0]}-{IMG_FRAMES[-1]} ({len(IMG_FRAMES)} frames):")
print("  CLEAR MOT:", {k: (round(v, 3) if isinstance(v, float) else v) for k, v in m_img.items() if k != "MOTP"},
      f"| MOTP = {m_img['MOTP']:.2f} box heights")
print("  HOTA:", {k: round(float(v), 3) for k, v in h_img.items()})
n_gt_ids = len({i for g in gts_i for i in g})
print(f"Text summary: {n_gt_ids} labelled objects, {m_img['GT']} object-frames; the tracker misses {m_img['FN']} "
      f"object-frames and adds {m_img['FP']} false ones; these drive MOTA ({m_img['MOTA']:.2f}). The {m_img['IDSW']} identity "
      f"switches are few by count, but each one hands the planner a new object without velocity.")

### Identity switches on real frames

An **identity switch** is a labelled object that the evaluation matches to one track ID and later to another. For each switch we look at the frames in between: was the object occluded, did the detector miss it, or did two tracks exchange identities without any missed frame?

In [ ]:
def idsw_events(log, frames):
    # per switch: labelled object, frame of the switch, old and new track ID, last frame with the old ID
    last, last_f, ev = {}, {}, []
    for f, mt in zip(frames, log):
        for gi, hi in mt.items():
            if gi in last and last[gi] != hi:
                ev.append(dict(obj=gi, frame=f, old=last[gi], new=hi, last_frame=last_f[gi]))
            last[gi], last_f[gi] = hi, f
    return pd.DataFrame(ev, columns=["obj", "frame", "old", "new", "last_frame"])


info = lab.set_index(["frame", "track_id"]).sort_index()
detected = det_eval.set_index(["frame", "track_id"])["detected"]
ev_img = idsw_events(log_i, IMG_FRAMES)
rows = []
for e in ev_img.itertuples():
    gap = list(range(e.last_frame + 1, e.frame))
    occ = [int(info.loc[(f, e.obj), "occluded"]) for f in gap if (f, e.obj) in info.index]
    hit = [bool(detected.get((f, e.obj), False)) for f in gap]
    if not gap:
        cause = "identity exchange with another track (no missed frame)"
    elif any(hit):
        cause = "detected but not associated (outside gate or assigned to another track)"
    elif occ and max(occ) >= 2:
        cause = "not detected while largely occluded"
    else:
        cause = "not detected while visible or partly occluded"
    rows.append({"object": e.obj, "class": info.loc[(e.frame, e.obj), "cls"], "switch at frame": e.frame,
                 "track IDs": f"#{e.old} -> #{e.new}", "detected in gap": f"{sum(hit)}/{len(hit)}",
                 "occlusion levels in gap": ",".join(map(str, occ)) or "-", "cause": cause})
sw_tab = pd.DataFrame(rows)
print(sw_tab.to_string(index=False) if len(sw_tab) else "No identity switches in this window.")

In [ ]:
# Show the switch with the longest gap: the object's labelled box, coloured and annotated with the track ID
# that the evaluation assigns to it in each frame
if len(ev_img):
    # prefer a switch caused by occlusion; otherwise take the one with the longest gap
    occl = sw_tab["cause"].str.contains("largely occluded").values
    cand = ev_img.assign(gap=ev_img["frame"] - ev_img["last_frame"], occl=occl)
    e = cand.sort_values(["occl", "gap"], ascending=False).iloc[0]
    show = sorted({max(IMG_FRAMES[0], int(e.last_frame) - 2), int(e.last_frame),
                   (int(e.last_frame) + int(e.frame)) // 2, int(e.frame), min(IMG_FRAMES[-1], int(e.frame) + 3)})
    g_rows = [info.loc[(f, int(e.obj))] for f in show]
    cx = np.mean([(g.x1 + g.x2) / 2 for g in g_rows]); cy_ = np.mean([(g.y1 + g.y2) / 2 for g in g_rows])
    fig, axes = plt.subplots(1, len(show), figsize=(3.4 * len(show), 4.2))
    for ax, f, g in zip(axes, show, g_rows):
        img = kt["images"][IMG_FRAMES.index(f)]
        x0, x1 = int(max(0, cx - 160)), int(min(img.shape[1], cx + 160)); y0, y1 = int(max(0, cy_ - 150)), int(min(img.shape[0], cy_ + 150))
        ax.imshow(img[y0:y1, x0:x1]); ax.grid(False); ax.set_xticks([]); ax.set_yticks([])
        tid = log_i[IMG_FRAMES.index(f)].get(int(e.obj))
        col = OI["blue"] if tid == e.old else OI["vermillion"] if tid == e.new else "white"
        ax.add_patch(Rect((g.x1 - x0, g.y1 - y0), g.x2 - g.x1, g.y2 - g.y1, fill=False, lw=2.5, color=col,
                          ls="-" if tid is not None else ":"))
        hit = bool(detected.get((f, int(e.obj)), False))
        ax.set_title(f"frame {f}\ntrack {'#' + str(tid) if tid is not None else 'none'} | occ {int(g.occluded)} | "
                     f"{'detected' if hit else 'missed'}", fontsize=9)
    plt.suptitle(f"Labelled {info.loc[(int(e.frame), int(e.obj)), 'cls']} {int(e.obj)}: track #{int(e.old)} (blue) is lost "
                 f"and the object returns as track #{int(e.new)} (vermillion)", fontweight="bold")
    plt.tight_layout(); plt.show()

Every such switch has the same consequence as in Section 6.4: the new track starts **without velocity** and must first be confirmed, so the planner briefly loses the object and then receives it with a poor motion estimate, exactly when it re-emerges from behind an occluder. With only 40 frames the counts are small; the next part uses the whole sequence.

### 9.5 The whole sequence in a world frame, with measured degradation

We now generate detections for all 376 frames from the **labelled 3D positions** (transformed into the world frame of Section 9.1), so that position quality is no longer the limiting factor, and degrade them with what we measured:

- each labelled object is detected with the **measured** $P_D$ for its class and occlusion level (Section 9.2, with a uniform Beta(1, 1) prior so that sparse cells stay defined);
- false positives arrive at the **measured** rate per frame, placed uniformly on the road ahead within 35 m;
- position noise is an **assumption** typical of a LiDAR or radar cluster centroid, $\sigma = 0.2\,\text{m} + 1\,\%$ of range per axis; the tracker is told the same $\mathbf{R}$.

Note that the occlusion level of a labelled object changes slowly, so the measured $P_D$ produces **runs** of missed detections behind occluders, unlike the independent dropouts of Section 8. We compare against ideal detections and vary the coasting limit $K_{\max}$. Evaluation is in metres (2 m gate, as in Section 6), within 35 m of the camera.

In [ ]:
MAX_RANGE = 35.0
cnt = det_eval.groupby(["cls", "occluded"])["detected"].agg(["sum", "size"])
pooled = det_eval.groupby("occluded")["detected"].agg(["sum", "size"])


def p_detect(cls, occ):
    s, n = cnt.loc[(cls, occ)] if (cls, occ) in cnt.index else pooled.loc[occ] if occ in pooled.index else (0, 0)
    return (s + 1) / (n + 2)


def sigma_pos(r):
    return 0.2 + 0.01 * r                        # assumed sensor noise per axis [m]


ign_lab3d = labels_all[labels_all["type"].isin(["Person_sitting", "Misc", "Tram"])]
ign_world = {f: list(rect_to_world(g[["x", "y", "z"]].values, f)) for f, g in ign_lab3d.groupby("frame")}
lab_by_frame = {f: g for f, g in lab.groupby("frame")}


def world_ground_truth(f):
    G = lab_by_frame.get(f, lab.iloc[:0])
    ok = ((G["y2"] - G["y1"]) >= 25) & (G["truncated"] <= 0.5) & (G["range_m"] <= MAX_RANGE)
    ev = {int(r.track_id): np.array([r.wx, r.wy]) for r in G[ok].itertuples()}
    return ev, [np.array([r.wx, r.wy]) for r in G[~ok].itertuples()] + ign_world.get(f, [])


def drop_on_ignored_world(hyp, gt, ign, thr=2.0):
    return {k: v for k, v in hyp.items() if any(np.linalg.norm(v - g) <= thr for g in gt.values())
            or not any(np.linalg.norm(v - g) <= thr for g in ign)}


def world_detections(f, r, ideal):
    G = lab_by_frame.get(f, lab.iloc[:0])
    G = G[G["range_m"] <= MAX_RANGE + 5]
    pos, cls, Rs = [], [], []
    for g in G.itertuples():
        if not ideal and r.random() > p_detect(g.cls, g.occluded):
            continue
        s = 0.05 if ideal else sigma_pos(g.range_m)
        pos.append(np.array([g.wx, g.wy]) + s * r.standard_normal(2)); cls.append(g.cls); Rs.append(s**2 * np.eye(2))
    if not ideal:
        for _ in range(r.poisson(FP_PER_FRAME)):
            p = np.array([r.uniform(-10, 10), CAM_H, r.uniform(5, MAX_RANGE)])
            s = sigma_pos(np.hypot(p[0], p[2]))
            pos.append(rect_to_world(p, f)[0]); cls.append(str(r.choice(["pedestrian", "vehicle"]))); Rs.append(s**2 * np.eye(2))
    return pos, cls, Rs


def time_to_confirm_real(log, gts, frames):
    # per labelled pedestrian: seconds from its first evaluated frame to its first match by a confirmed track
    first_seen, first_hit = {}, {}
    for f, g, mt in zip(frames, gts, log):
        for i in g:
            first_seen.setdefault(i, f)
        for i in mt:
            first_hit.setdefault(i, f)
    peds = [i for i in first_seen if info.loc[(first_seen[i], i), "cls"] == "pedestrian"]
    return np.array([(first_hit.get(i, frames[-1] + 1) - first_seen[i]) * 0.1 for i in peds])


ALL_FRAMES = list(range(n_frames))
gt_w = [world_ground_truth(f) for f in ALL_FRAMES]
gts_w = [g for g, _ in gt_w]
t_mc = time.time()
mc_rows = []
for setting, ideal, seeds in [("ideal", True, [0]), ("measured P_D and FP rate", False, [11, 12, 13])]:
    for k_max in [3, 8, 15]:
        for s in seeds:
            r = np.random.default_rng(s)
            Track._next_id = 0
            trk_w = MultiObjectTracker(dt=0.1, sigma_a=2.0, M=3, N=5, max_coast=k_max, fov=None)
            hy = []
            for f, (g, ign) in zip(ALL_FRAMES, gt_w):
                pos, cls, Rs = world_detections(f, r, ideal)
                o = trk_w.step(pos, cls, det_R=Rs)
                hy.append(drop_on_ignored_world({k: v[0][:2] for k, v in o.items()}, g, ign))
            lg = []
            mm, hh = clear_mot(gts_w, hy, log=lg), hota(gts_w, hy)
            ev = idsw_events(lg, ALL_FRAMES)
            ped_sw = int(sum(info.loc[(e.frame, e.obj), "cls"] == "pedestrian" for e in ev.itertuples()))
            ttc = time_to_confirm_real(lg, gts_w, ALL_FRAMES)
            mc_rows.append({"detections": setting, "K_max": k_max, "seed": s, "MOTA": mm["MOTA"], "HOTA": hh["HOTA"],
                            "AssA": hh["AssA"], "IDSW": mm["IDSW"], "pedestrian IDSW": ped_sw, "FP": mm["FP"],
                            "FN": mm["FN"], "ped. time-to-confirm, median [s]": np.median(ttc)})
mc = pd.DataFrame(mc_rows)
mc_sum = mc.groupby(["detections", "K_max"]).mean(numeric_only=True).drop(columns="seed")
print(f"{len(mc)} tracker runs over {n_frames} frames in {time.time() - t_mc:.1f} s "
      f"({sum(len(g) for g in gts_w)} evaluated object-frames, {len({i for g in gts_w for i in g})} objects)")
print(mc_sum.round(2).to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for setting, col in [("ideal", OI["blue"]), ("measured P_D and FP rate", OI["vermillion"])]:
    d = mc_sum.loc[setting]
    axes[0].plot(d.index, d["IDSW"], "o-", color=col, label=setting)
    axes[1].plot(d.index, d["FP"], "o-", color=col, label=setting)
    axes[2].plot(d.index, d["HOTA"], "o-", color=col, label=setting)
for ax, t, yl in zip(axes, ["Identity switches", "False-positive object-frames", "HOTA"],
                     ["IDSW [-]", "FP [object-frames]", "HOTA [-]"]):
    ax.set(title=t, xlabel="coasting limit K_max [frames]", ylabel=yl); ax.legend(fontsize=8)
plt.suptitle(f"KITTI sequence {SEQ:04d}, all {n_frames} frames (mean over seeds)", fontweight="bold")
plt.tight_layout(); plt.show()

ideal8, real8 = mc_sum.loc[("ideal", 8)], mc_sum.loc[("measured P_D and FP rate", 8)]
real3, real15 = mc_sum.loc[("measured P_D and FP rate", 3)], mc_sum.loc[("measured P_D and FP rate", 15)]
print(f"Text summary (K_max = 8): ideal detections give {ideal8['IDSW']:.1f} identity switches and HOTA {ideal8['HOTA']:.2f}; "
      f"with the measured detection rates {real8['IDSW']:.1f} switches ({real8['pedestrian IDSW']:.1f} on pedestrians) "
      f"and HOTA {real8['HOTA']:.2f}. Raising K_max from 3 to 15 changes the switches from {real3['IDSW']:.1f} to "
      f"{real15['IDSW']:.1f} and the false-positive object-frames from {real3['FP']:.0f} to {real15['FP']:.0f}. "
      f"Median pedestrian time-to-confirm: {ideal8['ped. time-to-confirm, median [s]']:.1f} s (ideal) vs "
      f"{real8['ped. time-to-confirm, median [s]']:.1f} s (measured).")

**Reading the real-data results.**

- With **ideal** detections the remaining identity switches and false positives come from the tracker itself: objects that leave the evaluated region and come back, pedestrians walking within a gate width of each other, and tracks that coast on after the label ends. Even here, longer coasting buys continuity with ghost tracks, the trade-off of Section 6.4.
- With the **measured** detection rates, the dominant error source is **occlusion**: largely occluded pedestrians are almost never detected by this detector, so in a crowded crossing tracks die behind other pedestrians and parked cars and are re-born under a new ID, which also delays confirmation. Longer coasting bridges some of these gaps, but every extra frame of coasting also keeps ghost tracks alive after objects have left; on this sequence the false-positive count grows much faster than the switch count falls.
- None of this is visible in a per-frame detection metric. A safety argument about a crossing pedestrian needs the **track-level** quantities: time-to-confirm, identity continuity through occlusion and the false-track rate, measured on data that contains the relevant occlusion patterns.

**Limits of this analysis.** One sequence and 40 detector frames are a demonstration, not a validation: the measured $P_D$ rests on a few hundred labelled boxes, the noise model in Section 9.5 is assumed, and the point-distance HOTA is an approximation of the official box-based metric (use TrackEval for reportable numbers).

## 10. Try it yourself (optional interactive widget)

The cell below first renders a static figure with default values, so it works in any viewer. If `ipywidgets` and a widget-capable front end are available (Jupyter, Colab), sliders appear that re-run the scene. Otherwise the cell degrades gracefully and you can call `explore(...)` directly with your own values.

In [ ]:
def explore(p_d=0.9, sigma_z=0.3, clutter_rate=1.0, max_coast=8, M=3, seed=7):
    fr = simulate_scene(p_d=p_d, sigma_z=sigma_z, clutter_rate=clutter_rate, seed=int(seed))
    out, _ = run_tracker(fr, max_coast=int(max_coast), M=int(M), N=5)
    g_, h_ = to_eval_frames(fr, out)
    mm, hh = clear_mot(g_, h_), hota(g_, h_)
    plt.figure(figsize=(13, 5.5))
    plot_scene(fr, out, title=f"P_D={p_d:.2f}, sigma_z={sigma_z:.2f} m, clutter={clutter_rate:.1f}/frame, "
                              f"K_max={int(max_coast)}, M={int(M)}/N=5")
    plt.tight_layout(); plt.show()
    print(f"MOTA={mm['MOTA']:.3f}  HOTA={hh['HOTA']:.3f}  IDSW={mm['IDSW']}  FP={mm['FP']}  FN={mm['FN']}  "
          f"pedestrian time-to-confirm={time_to_confirm(fr, out, 0):.1f} s")


explore()   # static default, always rendered

try:
    import ipywidgets as widgets
    widgets.interact(explore,
                     p_d=widgets.FloatSlider(value=0.9, min=0.3, max=1.0, step=0.05, description="P_D"),
                     sigma_z=widgets.FloatSlider(value=0.3, min=0.05, max=1.5, step=0.05, description="sigma_z [m]"),
                     clutter_rate=widgets.FloatSlider(value=1.0, min=0.0, max=10.0, step=0.5, description="clutter"),
                     max_coast=widgets.IntSlider(value=8, min=1, max=40, description="K_max"),
                     M=widgets.IntSlider(value=3, min=1, max=5, description="M (of 5)"),
                     seed=widgets.IntSlider(value=7, min=0, max=50, description="seed"))
except Exception as exc:            # no ipywidgets or no widget front end
    print(f"Interactive sliders unavailable ({type(exc).__name__}); call explore(p_d=..., sigma_z=...) instead.")


## 11. Beyond the linear constant-velocity filter

- **Nonlinear motion and measurements.** Vehicles are better described by coordinated-turn or bicycle-type models (e.g. constant turn rate and velocity, CTRV), and radar measures range, bearing and range rate. The **extended Kalman filter (EKF)** linearises the models with Jacobians; the **unscented Kalman filter (UKF)** propagates a deterministic set of sigma points through the nonlinear functions and is often more accurate for strongly nonlinear models (Julier and Uhlmann, 2004).
- **Manoeuvre switching.** Road users switch between regimes (cruising, braking, turning, stopping). The **interacting multiple model (IMM)** estimator runs several filters in parallel and mixes them according to a Markov chain over modes (Blom and Bar-Shalom, 1988). It addresses exactly the mismatch our decelerating car exposes.
- **Tracking-by-detection with learned components.** SORT (Bewley et al., 2016) is a Kalman filter plus Hungarian assignment on image boxes, essentially Sections 2–4 of this notebook; DeepSORT (Wojke et al., 2017) adds a learned appearance descriptor to the association cost, which reduces identity switches through occlusions. In 3D, AB3DMOT (Weng et al., 2020) showed that a carefully engineered Kalman-filter baseline is competitive on KITTI and nuScenes, and CenterPoint (Yin et al., 2021) tracks by greedy matching using a learned velocity estimate.
- **End-to-end learned trackers.** Transformer-based methods such as MOTR (Zeng et al., 2022) propagate *track queries* across frames and learn association implicitly. They can be strong on benchmarks but give up the explicit covariance and the analysable gating/management logic that a safety argument can reason about, which raises the verification questions discussed in [Module 06](../06_AI_Safety_ISO_PAS_8800/01_ai_safety_standards.ipynb).

Whatever the architecture, the safety-relevant questions remain those of this notebook: Is identity preserved through occlusions and reclassification? Is the reported uncertainty consistent? How long does it take to confirm a new object under degraded detection?

## 12. Exercises

**Exercise 1 (filter tuning).** Re-run Section 2.5 with $\sigma_{a,\text{true}} = 2$ m/s² but keep the matched filter at $\sigma_a = 0.5$. What happens to NEES and NIS? Which of the two could you monitor in a vehicle?

<details><summary>Answer</summary>

The filter becomes over-confident: both average NEES and NIS rise above their 95 % bands because the real manoeuvres exceed what $\mathbf{Q}$ allows. Only **NIS** can be monitored online, because it requires no ground truth; a sustained NIS above the $\chi^2_2$ band is a runtime indicator that the motion or noise model no longer matches reality.
</details>

**Exercise 2 (confirmation logic).** Using `confirm_latency_mc`, find the smallest $M$ (with $N=5$) for which the 95th-percentile time-to-confirm stays below 0.5 s at $P_D = 0.7$. What does a smaller $M$ cost?

<details><summary>Answer</summary>

Compute `np.percentile(confirm_latency_mc(0.7, M, 5), 95) * 0.1` for $M = 1, \dots, 5$. With $M=2$ the latency is typically within a few frames, whereas $M=4$ exceeds 0.5 s. A smaller $M$ confirms clutter more easily: re-run the scene with `clutter_rate=6` and compare the number of false-positive frames for $M=2$ and $M=4$. The choice must be argued against both hazards: a late reaction to a real object, and unnecessary braking for a ghost.
</details>

**Exercise 3 (class-aware gating).** Some trackers only associate detections with tracks of the *same* class. Implement this in `associate` (set the cost to `BIG` when classes differ) and repeat Section 7. Is the result closer to "retain history" or to "reset on reclassification"? Why?

<details><summary>Answer</summary>

Closer to "reset": a detection with a flipped label cannot update the existing track, so the track misses that frame and a new tentative track is spawned for the detection. Class-aware gating is useful against swaps between nearby objects of different classes, but it should be combined with tolerance to class confusion, for instance by gating on a class *distribution* or allowing confusable classes (pedestrian/cyclist) to associate.
</details>

**Exercise 4 (safety requirement).** Write a quantitative, verifiable requirement on the tracker that would have addressed the issue described in Section 1. State the metric, the threshold and the conditions under which it must hold.

<details><summary>Answer (one possible formulation)</summary>

*"For any object within 60 m of the ego path, a change of the object's classification shall not reset its track identity or kinematic history. Over the validation scenario set covering pedestrians, cyclists and pedestrians pushing bicycles crossing outside crosswalks, by day and night, the confirmed track shall be maintained with at most one identity switch per object, and the velocity estimate shall be within 0.5 m/s within 1.0 s of first detection in at least 99 % of trials."* The thresholds must be derived from the planner's needs (stopping distance at the ODD's maximum speed) and justified in the SOTIF argument.
</details>

**Exercise 5 (real data).** In Section 9.5, set the coasting limit to $K_{\max}=25$ and, separately, replace the measured $P_D$ by the value for fully visible objects for every occlusion level. Which change reduces pedestrian identity switches more, and what does each cost?

<details><summary>Answer</summary>

Longer coasting bridges some occlusions but adds ghost-track frames after objects leave (FP rises steeply). Removing the occlusion penalty on $P_D$ (i.e. a detector or sensor set that still sees largely occluded pedestrians, such as an elevated or second viewpoint, LiDAR, or V2X) removes the cause of most switches without creating ghosts. The comparison shows why occlusion is treated as a triggering condition to be addressed in perception and sensor design, not only by tuning the tracker.
</details>

## In industry

Tracking sits between perception and planning, so its evidence is reviewed from both sides. The results of this notebook map onto these work products:

| Work product | Typical owner | What this notebook contributes |
|---|---|---|
| **Tracker requirements and interface specification** (object list: position, velocity, covariance, existence probability, class distribution, track age) | Perception / sensor-fusion architect | Track-level acceptance criteria such as time-to-confirm, identity continuity through occlusion, maximum ghost-track duration, covariance consistency (NEES/NIS) |
| **SOTIF triggering-condition analysis** (ISO 21448:2022, Clauses 6–7) | SOTIF engineer with perception experts | Occlusion, low $P_D$ and unmodelled noise as triggering conditions, with their measured effect on track-level metrics (Sections 8 and 9) |
| **Perception KPI and V&V report** | Validation engineer | MOTA, HOTA (DetA/AssA), IDSW, FP/FN and time-to-confirm per class, range band and occlusion level, on recorded and simulated data |
| **Runtime monitoring concept** | Safety architect | NIS-based consistency monitoring and track-quality flags passed to planning |

**What reviewers look for**

- Metrics are reported **per class, range band and occlusion level**, not only as one aggregate; pedestrians and cyclists near the ego path are shown separately.
- Track-level criteria (time-to-confirm, IDSW per object, ghost duration) are **derived from planner needs**, e.g. stopping distance at the ODD's maximum speed, not chosen for convenience.
- The ground-truth tracks, ignore regions and matching thresholds of the evaluation are documented, and the official metric code (e.g. TrackEval) is used for reported numbers.
- Tuning parameters ($\sigma_a$, $\mathbf{R}$, gate, $M$-of-$N$, $K_{\max}$) are under configuration control with a rationale, and changes trigger regression tests on a fixed scenario set.
- Robustness to **reclassification** is shown explicitly (the Section 1 lesson), and covariance consistency is checked on real data, not only in simulation.


## Key takeaways

- Tracking turns per-frame detections into objects with **identity, velocity and uncertainty**; planning depends on all three.
- The **Kalman filter** is a predict–update recursion on a mean and covariance. Its value for safety lies as much in the covariance as in the estimate, so **consistency** (NEES, NIS) must be checked, and over-confidence is the dangerous direction.
- **Gating** with the Mahalanobis distance and **global assignment** (Hungarian) are simple, analysable association steps; greedy matching can be measurably worse.
- **Track management** parameters (M-of-N, coasting limit) trade late confirmation against false tracks, and identity loss against ghost objects. They must be justified from the ODD.
- **MOTA** is dominated by detection errors; **HOTA** balances detection and association; **AMOTA** removes the dependence on one detector threshold. Report several.
- On real data (KITTI 0015) a pretrained detector finds fully visible pedestrians reliably but largely occluded ones almost never; the resulting **runs of missed detections** cause most identity switches and delay confirmation. Track-level metrics, not per-frame detection metrics, expose this.
- Lower $P_D$ and higher noise, typical effects of **SOTIF triggering conditions**, lengthen time-to-confirm and fragment tracks. Discarding history on reclassification, as described in the NTSB report on the 2018 Tempe crash, turns classification uncertainty into a failure to predict motion.

## Self-assessment

1. Why can a single detector frame not provide an object's velocity, and why does that matter for a crossing pedestrian?
2. Write down the predict and update equations of the Kalman filter and explain the role of $\mathbf{Q}$ and $\mathbf{R}$.
3. What does an average NEES persistently above the upper $\chi^2$ bound indicate, and why is that worse than being below the lower bound?
4. Why is the Mahalanobis gate size different for a track that has just been updated and one that has coasted for one second?
5. Explain the trade-off encoded by M-of-N confirmation and by the coasting limit $K_{\max}$.
6. Give one reason why MOTA alone is an insufficient tracking metric.
7. Name two triggering conditions that would lower $P_D$ for a pedestrian, and describe their effect on time-to-confirm.
8. Why does Section 9 transform the KITTI labels into a world frame before tracking, and why are monocular ground-plane ranges not used for the BEV analysis?

<details><summary>Answers</summary>

1. Velocity is a time derivative and is not observable from one position measurement; at least two associated measurements are needed, and a filter needs several to reduce noise. Without velocity, the planner cannot predict whether the pedestrian will enter the ego path.
2. Predict: $\hat{\mathbf{x}}^- = \mathbf{F}\hat{\mathbf{x}}$, $\mathbf{P}^- = \mathbf{F}\mathbf{P}\mathbf{F}^\top + \mathbf{Q}$. Update: $\mathbf{S}=\mathbf{H}\mathbf{P}^-\mathbf{H}^\top+\mathbf{R}$, $\mathbf{K}=\mathbf{P}^-\mathbf{H}^\top\mathbf{S}^{-1}$, $\hat{\mathbf{x}}=\hat{\mathbf{x}}^-+\mathbf{K}(\mathbf{z}-\mathbf{H}\hat{\mathbf{x}}^-)$, $\mathbf{P}=(\mathbf{I}-\mathbf{K}\mathbf{H})\mathbf{P}^-$. $\mathbf{Q}$ expresses how far the true motion may deviate from the model (manoeuvres); $\mathbf{R}$ expresses sensor noise. Their ratio sets how strongly the filter follows new measurements.
3. Over-confidence: the covariance is smaller than the actual error. Gates become too narrow (missed associations, track loss) and downstream consumers trust wrong positions. Under-confidence is wasteful but errs on the safe side.
4. $\mathbf{S} = \mathbf{H}\mathbf{P}\mathbf{H}^\top + \mathbf{R}$ grows with $\mathbf{P}$, which grows with every prediction step without an update. A coasted track has a larger gate.
5. Larger $M$ suppresses clutter but delays confirmation of real objects; larger $K_{\max}$ bridges occlusions and preserves identity but keeps ghost tracks alive.
6. It counts identity switches only once per switch, whereas FN and FP accumulate per frame, so it mainly reflects detection quality; it also depends on the detector threshold (motivating AMOTA) and ignores how long identities are kept (motivating HOTA's AssA).
7. For example: heavy rain or spray (LiDAR/camera degradation), low light or glare (camera), partial occlusion by parked vehicles, unusual appearance such as a pedestrian pushing a bicycle. Each lowers $P_D$ and lengthens the time until M of N detections accumulate (Section 4.1).
8. In the moving camera frame, ego braking and turning appear as object accelerations that violate the constant-velocity model; in the world frame, parked objects stand still. Flat-road ranging from the lower box edge has errors of many metres beyond about 20 m (pitch, road slope, occluded lower edges), so a BEV tracker fed with it would mainly measure the ranging error.
</details>

## References

- Bar-Shalom, Y., Li, X. R., Kirubarajan, T. (2001). *Estimation with Applications to Tracking and Navigation*. Wiley. https://doi.org/10.1002/0471221279
- Bernardin, K., Stiefelhagen, R. (2008). Evaluating multiple object tracking performance: the CLEAR MOT metrics. *EURASIP Journal on Image and Video Processing*, 2008, 246309. https://doi.org/10.1155/2008/246309
- Bewley, A., Ge, Z., Ott, L., Ramos, F., Upcroft, B. (2016). Simple online and realtime tracking. *IEEE ICIP 2016*, 3464–3468. https://doi.org/10.1109/ICIP.2016.7533003
- Blom, H. A. P., Bar-Shalom, Y. (1988). The interacting multiple model algorithm for systems with Markovian switching coefficients. *IEEE Transactions on Automatic Control*, 33(8), 780–783. https://doi.org/10.1109/9.1299
- Caesar, H., et al. (2020). nuScenes: a multimodal dataset for autonomous driving. *IEEE/CVF CVPR 2020*, 11618–11628. https://doi.org/10.1109/CVPR42600.2020.01164
- Geiger, A., Lenz, P., Stiller, C., Urtasun, R. (2013). Vision meets robotics: the KITTI dataset. *International Journal of Robotics Research*, 32(11), 1231–1237. https://doi.org/10.1177/0278364913491297
- Geiger, A., Lenz, P., Urtasun, R. (2012). Are we ready for autonomous driving? The KITTI vision benchmark suite. *IEEE CVPR 2012*, 3354–3361. https://doi.org/10.1109/CVPR.2012.6248074
- Howard, A., et al. (2019). Searching for MobileNetV3. *IEEE/CVF ICCV 2019*, 1314–1324. https://doi.org/10.1109/ICCV.2019.00140
- ISO 21448:2022. *Road vehicles — Safety of the intended functionality*. International Organization for Standardization.
- Julier, S. J., Uhlmann, J. K. (2004). Unscented filtering and nonlinear estimation. *Proceedings of the IEEE*, 92(3), 401–422. https://doi.org/10.1109/JPROC.2003.823141
- Kalman, R. E. (1960). A new approach to linear filtering and prediction problems. *Journal of Basic Engineering*, 82(1), 35–45. https://doi.org/10.1115/1.3662552
- Kuhn, H. W. (1955). The Hungarian method for the assignment problem. *Naval Research Logistics Quarterly*, 2(1–2), 83–97. https://doi.org/10.1002/nav.3800020109
- Lin, T.-Y., et al. (2014). Microsoft COCO: common objects in context. *Computer Vision – ECCV 2014*, LNCS 8693, 740–755. https://doi.org/10.1007/978-3-319-10602-1_48
- Luiten, J., Os̆ep, A., Dendorfer, P., Torr, P., Geiger, A., Leal-Taixé, L., Leibe, B. (2021). HOTA: a higher order metric for evaluating multi-object tracking. *International Journal of Computer Vision*, 129, 548–578. https://doi.org/10.1007/s11263-020-01375-2
- Munkres, J. (1957). Algorithms for the assignment and transportation problems. *Journal of the Society for Industrial and Applied Mathematics*, 5(1), 32–38. https://doi.org/10.1137/0105003
- National Transportation Safety Board (2019). *Collision Between Vehicle Controlled by Developmental Automated Driving System and Pedestrian, Tempe, Arizona, March 18, 2018*. Highway Accident Report NTSB/HAR-19/03 (accident ID HWY18MH010). https://www.ntsb.gov/investigations/AccidentReports/Reports/HAR1903.pdf
- Ren, S., He, K., Girshick, R., Sun, J. (2017). Faster R-CNN: towards real-time object detection with region proposal networks. *IEEE Transactions on Pattern Analysis and Machine Intelligence*, 39(6), 1137–1149. https://doi.org/10.1109/TPAMI.2016.2577031
- Weng, X., Wang, J., Held, D., Kitani, K. (2020). 3D multi-object tracking: a baseline and new evaluation metrics. *IEEE/RSJ IROS 2020*, 10359–10366. https://doi.org/10.1109/IROS45743.2020.9341164
- Wojke, N., Bewley, A., Paulus, D. (2017). Simple online and realtime tracking with a deep association metric. *IEEE ICIP 2017*, 3645–3649. https://doi.org/10.1109/ICIP.2017.8296962
- Yin, T., Zhou, X., Krähenbühl, P. (2021). Center-based 3D object detection and tracking. *IEEE/CVF CVPR 2021*, 11779–11788. https://doi.org/10.1109/CVPR46437.2021.01161
- Zeng, F., Dong, B., Zhang, Y., Wang, T., Zhang, X., Wei, Y. (2022). MOTR: end-to-end multiple-object tracking with transformer. *Computer Vision – ECCV 2022*, LNCS, 659–675. https://doi.org/10.1007/978-3-031-19812-0_38

**Related work by the author:** Patel, M., Jung, R. (2024). Simulation-based performance evaluation of 3D object detection methods with deep learning for a LiDAR point cloud dataset in a SOTIF-related use case. *VEHITS 2024*. https://doi.org/10.5220/0012707300003702

---
**Navigation:** ← [06 Sensor fusion](06_sensor_fusion.ipynb) · [Module index](README.md) · [08 Driving datasets](08_driving_datasets.ipynb) →